# Model Training & Experiments
## Apple vs Banana Image Classifier

**Project:** Apple vs Banana image classification  
**Dataset:** Dataset – 2,544 images (1,272 Apple / 1,272 Banana, 1,272 Student / 1,272 Public), 128 × 128, pixels 0–1  
**Goal:** Train and evaluate four predefined model configurations on every team computer and capture reproducible evidence for model comparison, reliability analysis and deployment  
**Design:** 4 models × 4 computers = 16 experimental runs  


This notebook implements:
- Dataset audit with leakage checks (problems are flagged, never silently fixed)
- One shared preprocessing pipeline and training-only augmentation
- Four predefined models: Custom CNN (baseline and tuned) and EfficientNetB0 (transfer learning and fine-tuned)
- Standard evaluation on the held-out test set (accuracy, precision, recall, F1, confusion matrices, learning curves)
- Error, confidence/reliability and student-vs-public analysis
- Cross-computer reproducibility records
- Stretch analyses: feature space, test-time augmentation, unsupported-input rejection, computational comparison
- Automatic export of every result into one consistent `results/` folder

## **Overview:**
 1. Experiment Definition
 2. Environment & Reproducibility
 3. Dataset & Data Integrity Audit
 4. Data Loading & Preprocessing
 5. Training Augmentation
 6. Model Configuration
 7. Model 1 – Custom CNN Baseline
 8. Model 2 – Custom CNN Tuned
 9. Model 3 – EfficientNetB0 Transfer Learning
 10. Model 4 – EfficientNetB0 Fine-Tuned
 11. Training & Experiment Logging
 12. Standard Model Evaluation (metrics, confusion matrices, learning curves, student vs public)
 13. Cross-Computer Reproducibility
 14. Error Analysis
 15. Confidence & Reliability Analysis
 16. Feature-Space Analysis [STRETCH]
 17. Test-Time Augmentation [STRETCH]
 18. Unsupported-Input / OOD Analysis [STRETCH]
 19. Computational Comparison [STRETCH]
 20. Save & Export All Results
 21. Final Experiment Summary

## 1. Experiment Definition

### 1.1 What this experiment tests
Each of **four models** is trained independently on **four computers** (PC1–PC4), giving **16 experimental runs**. The experiment is not simply "we trained four models" – it is:

> Each of four models was independently executed across four computing environments, on the same dataset, with the same code, seed and configuration.

We are testing **reproducibility/consistency across environments**. We are *not* claiming that four computers make a model more robust – differences between computers can appear because the hardware/software environments differ, and this notebook records the evidence needed to investigate them.

| ID | Model | Purpose |
|----|-------|---------|
| M1 | Custom CNN Baseline | Baseline trained from scratch |
| M2 | Custom CNN Tuned | Does controlled optimisation improve the baseline? |
| M3 | EfficientNetB0 Transfer Learning | Pretrained features + new classification head |
| M4 | EfficientNetB0 Fine-Tuned | Does adapting later pretrained layers help? |

### 1.2 Dataset (Dataset)
| Property | Value |
|----------|-------|
| Total images | 2,544 |
| Apple / Banana | 1,272 / 1,272 |
| Student / Public | 1,272 / 1,272 |
| Image size | 128 × 128 |
| Pixel range | 0–1 |
| Batch size | 32 |

#### Split
| Split | Images | Apple | Banana |
|-------|--------|-------|--------|
| Training | 1,780 | 890 | 890 |
| Validation | 382 | 190 | 192 |
| Test | 382 | 192 | 190 |

**The test set is held out from model training and hyperparameter selection.** Early stopping and (optional) tuning use the validation set only; each model is evaluated on the test set once, after training has finished.

### 1.3 How to run this notebook (read first)
1. Open Section 2.2 and change **only** `COMPUTER_ID` (and `DATASET_ROOT` if your dataset folder has a different relative path). Do not edit any model, training or configuration code – the point of the experiment is that every computer runs identical code.
2. Plug the laptop in and disable sleep. A sleeping laptop keeps the wall-clock timer running, which distorts training times.
3. Run all cells top to bottom (Kernel → Restart & Run All). Progress is printed every epoch.
4. When finished, save the executed notebook as `Model_Training_and_Experiments_PCn.ipynb` (where n is your own PC number) and send it together with the `results_PCn.zip` file created in Section 20.

In [ ]:
# Dataset specification 
DATASET_VERSION = "Apple_Banana_Dataset_(Final)"
NOTEBOOK_VERSION = "v1.0"          
PREPROCESSING_VERSION = "PP_v1: EXIF-upright, RGB, bicubic resize 128x128, pixels 0-1"

CLASSES = ["Apple", "Banana"]          # ImageFolder sorts alphabetically: 0 = Apple, 1 = Banana
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42
CONFIDENCE_THRESHOLD = 0.90            # Confidence gate used in Sections 15 and 18

# Expected dataset composition (checked automatically in Section 3)
EXPECTED = {
    "total": 2544, "Apple": 1272, "Banana": 1272, "Student": 1272, "Public": 1272,
    "train": 1780, "validation": 382, "test": 382,
    "train_Apple": 890, "train_Banana": 890,
    "validation_Apple": 190, "validation_Banana": 192,
    "test_Apple": 192, "test_Banana": 190,
}

# Experiment design: 4 models x 4 computers = 16 runs
MODEL_IDS = ["M1", "M2", "M3", "M4"]
COMPUTER_IDS = ["PC1", "PC2", "PC3", "PC4"]

# Locked switches (identical on every computer)
STOP_ON_AUDIT_FAIL = False     # False = flag audit problems but keep running; True = stop the notebook on a failed audit
APPLY_EXIF_TRANSPOSE = True    # Phone photos store their rotation in EXIF; apply it so every image is upright
NEAR_DUP_MAX_HAMMING = 4       # Perceptual-hash distance (out of 64 bits) that counts as a near-duplicate

print("=== Experiment definition ===")
print(f"Models    : {len(MODEL_IDS)} ({', '.join(MODEL_IDS)})")
print(f"Computers : {len(COMPUTER_IDS)} ({', '.join(COMPUTER_IDS)})")
print(f"Total runs: {len(MODEL_IDS) * len(COMPUTER_IDS)}")
print(f"Dataset   : {DATASET_VERSION}, {EXPECTED['total']:,} images, image size {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}, batch size {BATCH_SIZE}")
print(f"Split     : train {EXPECTED['train']:,} / validation {EXPECTED['validation']} / test {EXPECTED['test']}")

## 2. Environment & Reproducibility
This section identifies the computer running the notebook and fixes everything that could make two runs differ for reasons other than the hardware: the random seed, the dataset version, the notebook version and the configuration hash (Section 6).

### 2.1 Imports

In [ ]:
import os
import io
import re
import gc
import sys
import glob
import json
import time
import copy
import math
import random
import shutil
import hashlib
import getpass
import platform
import socket
import subprocess
import datetime
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import transforms, datasets, models
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

import sklearn
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score, roc_curve
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from sklearn.decomposition import PCA
from sklearn.covariance import LedoitWolf

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):     # Fallback when tqdm is not installed
        return iterable

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
print("Imports Done")

### 2.2 Your configuration (the only cell you edit)

In [ ]:
# =================================================================================
# EDIT ONLY THIS CELL - everything else must stay identical on all four computers
# =================================================================================
COMPUTER_ID = "PCn"                                # PC1, PC2, PC3 or PC4
DATASET_ROOT = "Apple_Banana_Dataset_(Final)"      # Relative path to the dataset folder

# Optional run switches
QUICK_TEST = False        # True = ~5 minute smoke test on a small subset (saved to results_QUICKTEST, NOT a valid run)
RUN_STRETCH = True        # False skips Sections 16-19 (feature space, TTA, unsupported input, computational comparison)
RESUME = True             # True = skip models that already finished in this results folder (safe after an interruption)
RUN_TUNING_SWEEP = False  # PC1 already locked M2 via the sweep - leave this False so PC2/PC3/PC4 just load the lock
OVERWRITE_LOCK = False    # True lets RUN_TUNING_SWEEP replace an existing M2_locked_config.json - see Section 11.0

### 2.3 Reproducibility settings
The seed is reset at the start of every model (Section 11), so the result of M2 does not depend on whether M1 ran first. GPU kernels and different CPU thread counts can still change floating-point rounding slightly – that is exactly the kind of cross-computer difference Section 13 records.

In [ ]:
if COMPUTER_ID not in COMPUTER_IDS:
    raise ValueError(f"COMPUTER_ID must be one of {COMPUTER_IDS}, got '{COMPUTER_ID}'")

# Set random seeds for reproducibility
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CPU threads: use physical cores (oversubscribing hyper-threads slows PyTorch down on laptops)
try:
    import psutil
    N_PHYS = psutil.cpu_count(logical=False) or os.cpu_count() or 1
except ImportError:
    psutil = None
    N_PHYS = max(1, (os.cpu_count() or 2) // 2)
torch.set_num_threads(N_PHYS)
torch.set_flush_denormal(True)      # Denormal floats can make CPU training much slower

# Results folders (relative paths, one file per experiment: M1_PC1, M1_PC2, ...)
RESULTS_ROOT = "results_QUICKTEST" if QUICK_TEST else "results"
SUBDIRS = ["metrics", "confusion_matrices", "learning_curves", "predictions", "embeddings",
           "experiment_logs", "reproducibility", "checkpoints", "error_analysis", "reliability",
           "tta", "ood", "computational", "audit"]
DIRS = {name: f"{RESULTS_ROOT}/{name}" for name in SUBDIRS}
for path in DIRS.values():
    os.makedirs(path, exist_ok=True)

def exp_id(model_id):
    """Experiment ID = Model ID + Computer ID, e.g. M1_PC2"""
    return f"{model_id}_{COMPUTER_ID}"

def save_fig(fig, subdir, name, show=True):
    """Save a figure into the results folder and (optionally) show it in the notebook."""
    fig.savefig(f"{DIRS[subdir]}/{name}.png", dpi=120, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)

print(f"Computer ID : {COMPUTER_ID}")
print(f"Device      : {device}  |  CPU threads: {torch.get_num_threads()}")
print(f"Results in  : {RESULTS_ROOT}/")
if QUICK_TEST:
    print("\n*** QUICK_TEST is ON: small subset, 2 epochs per model. These results are a smoke test, NOT a valid experimental run. ***")

### 2.4 Environment capture

In [ ]:
def gpu_info():
    if not torch.cuda.is_available():
        return {"available": False}
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            stderr=subprocess.DEVNULL
        ).decode().strip().splitlines()
        gpus = []
        for line in out:
            name, mem, driver = [x.strip() for x in line.split(",")]
            gpus.append({"name": name, "memory": mem, "driver": driver})
        return {"available": True, "devices": gpus, "count": len(gpus)}
    except Exception as e:
        return {"available": True, "name": torch.cuda.get_device_name(0), "error": str(e)}

def cpu_info():
    info = {"model": platform.processor() or "unknown", "cores_logical": os.cpu_count(), "cores_physical": N_PHYS}
    try:
        if platform.system() == "Linux":
            with open("/proc/cpuinfo") as f:
                for line in f:
                    if line.lower().startswith("model name"):
                        info["model"] = line.split(":", 1)[1].strip()
                        break
        elif platform.system() == "Darwin":
            info["model"] = subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"],
                                                    stderr=subprocess.DEVNULL).decode().strip()
        elif platform.system() == "Windows":
            out = subprocess.check_output(["wmic", "cpu", "get", "name"], stderr=subprocess.DEVNULL).decode()
            names = [l.strip() for l in out.splitlines() if l.strip() and l.strip().lower() != "name"]
            if names:
                info["model"] = names[0]
    except Exception:
        pass
    return info

def ram_gb():
    if psutil is not None:
        return round(psutil.virtual_memory().total / 1e9, 1)
    return None

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return None

ENV_INFO = {
    "computer_id":          COMPUTER_ID,
    "hostname":             socket.gethostname(),
    "user":                 getpass.getuser(),
    "timestamp":            datetime.datetime.now().isoformat(timespec="seconds"),
    "os":                   f"{platform.system()} {platform.release()}",
    "python":               platform.python_version(),
    "pytorch":              torch.__version__,
    "torchvision":          torchvision.__version__,
    "numpy":                np.__version__,
    "pandas":               pd.__version__,
    "scikit_learn":         sklearn.__version__,
    "pillow":               Image.__version__ if hasattr(Image, "__version__") else "unknown",
    "cuda_version":         torch.version.cuda,
    "cudnn_version":        torch.backends.cudnn.version(),
    "device_used":          str(device),
    "cpu":                  cpu_info(),
    "cpu_threads_used":     torch.get_num_threads(),
    "gpu":                  gpu_info(),
    "ram_gb":               ram_gb(),
    "seed":                 SEED,
    "deterministic_cudnn":  torch.backends.cudnn.deterministic,
    "dataset_version":      DATASET_VERSION,
    "notebook_version":     NOTEBOOK_VERSION,
    "preprocessing_version": PREPROCESSING_VERSION,
    "git_commit":           git_commit(),
    "quick_test":           QUICK_TEST,
    "dataset_fingerprint":  None,    # Filled in Section 3
    "config_hash":          None,    # Filled in Section 6
}

gpu_text = ", ".join(g["name"] for g in ENV_INFO["gpu"].get("devices", [])) or ENV_INFO["gpu"].get("name", "none (CPU only)")
env_table = pd.DataFrame({
    "Item": ["Computer ID", "OS", "Python", "Framework", "CPU", "CPU cores (physical / logical)", "GPU", "RAM (GB)",
             "CUDA version", "Random seed", "Dataset version", "Notebook version", "Execution date/time"],
    "Value": [COMPUTER_ID, ENV_INFO["os"], ENV_INFO["python"], f"PyTorch {torch.__version__} / torchvision {torchvision.__version__}",
              ENV_INFO["cpu"]["model"], f"{N_PHYS} / {os.cpu_count()}", gpu_text, ENV_INFO["ram_gb"],
              ENV_INFO["cuda_version"] or "n/a", SEED, DATASET_VERSION, NOTEBOOK_VERSION, ENV_INFO["timestamp"]],
})
print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print(env_table.to_string(index=False))

## 3. Dataset Verification
Full dataset auditing and integrity checking - file counts, corrupted or duplicate images, and cross-split leakage from augmented copies - was carried out earlier in the project, during **Pillar 1: Dataset & Data Engineering**, not here.  For the record, the final Pillar 1 audit reported:

- Images that are augmented copies (tag in filename): 1,272/2,544 - tags: `vertical_flip` 211, `rotated_by_75` 198, `rotated_by_45` 182, `translation` 160, `rotated_by_15` 159, `rotated_by_30` 156
- Photos with several copies (identical pixels / augmented / same name): 2 / 350 / 0
- Duplicate leakage across splits (identical pixels): 0
- Augmented copies of the same photo across splits: 0
- Test images with an identical/augmented copy in train or validation: 0
- Validation images with an identical/augmented copy in train: 0
- Same original filename across splits (no tag): 0 (checked - different photos)
- Student/public overlap (identical pixels): 0

This section does not repeat that work. It only checks that the metadata and the files actually on disk still agree - a cheap guard against the folder being swapped, partially copied, or edited since Pillar 1 signed off, not a re-run of the audit above.

In [ ]:
def find_subdir(parent, name):
    """Return the sub-folder of `parent` whose name matches `name` ignoring letter case."""
    for d in os.listdir(parent):
        if d.lower() == name.lower() and os.path.isdir(os.path.join(parent, d)):
            return os.path.join(parent, d)
    raise FileNotFoundError(f"Folder '{name}' not found inside '{parent}'")

def parse_disk_filename(fn):
    """Return (source_type, image_id) from a disk filename like PUBLIC_PUB_0001_<original name>."""
    stem = os.path.splitext(str(fn))[0]
    m = re.match(r"^(PUBLIC|STUDENT)_([A-Za-z]+_\d+)_", stem)
    if m:
        return m.group(1).capitalize(), m.group(2)
    return "Unknown", None

if not os.path.isdir(DATASET_ROOT):
    raise FileNotFoundError(f"Dataset folder '{DATASET_ROOT}' not found. Current working folder: {os.getcwd()}")

SPLIT_DIRS = {split: find_subdir(DATASET_ROOT, split) for split in ["train", "validation", "test"]}
META_PATH = os.path.join(find_subdir(DATASET_ROOT, "Metadata"), "final_metadata.csv")
meta = pd.read_csv(META_PATH)
meta["key"] = meta["image_id"].astype(str).str.strip().str.lower()

rows = []
for split, split_dir in SPLIT_DIRS.items():
    ds = datasets.ImageFolder(root=split_dir)
    for path, label_idx in ds.samples:
        path = os.path.normpath(path)
        fn = os.path.basename(path)
        source, image_id = parse_disk_filename(fn)
        label = ds.classes[label_idx]
        rows.append({"path": path, "filename": fn, "split": split, "label": label,
                     "label_idx": CLASSES.index(label) if label in CLASSES else -1,
                     "source": source, "image_id": image_id})
inv = pd.DataFrame(rows)
inv["key"] = inv["image_id"].astype(str).str.strip().str.lower()

# Metadata <-> disk agreement - NOT a re-run of the Pillar 1 audit, just "does this folder match its own metadata"
merged = inv.merge(meta[["key", "label", "split", "source_type"]], on="key", how="outer", suffixes=("", "_csv"), indicator=True)
both = merged[merged["_merge"] == "both"]
checks = {
    "Total images":                           (EXPECTED["total"], len(inv)),
    "On disk but missing from metadata":       (0, int((merged["_merge"] == "left_only").sum())),
    "In metadata but missing from disk":       (0, int((merged["_merge"] == "right_only").sum())),
    "Label mismatches (folder vs metadata)":   (0, int((both["label"].str.lower() != both["label_csv"].astype(str).str.lower()).sum())),
    "Split mismatches (folder vs metadata)":   (0, int((both["split"].str.lower() != both["split_csv"].astype(str).str.lower()).sum())),
}
print("DATASET VERIFICATION (metadata vs disk)")
print("=" * 50)
for name, (expected, actual) in checks.items():
    print(f"{'O' if actual == expected else 'X'} {name}: {actual}  (expected {expected})")

AUDIT_STATUS = "PASS" if all(actual == expected for expected, actual in checks.values()) else "FAIL"
print(f"\nResult: {AUDIT_STATUS}")
if AUDIT_STATUS == "FAIL":
    print("WARNING!!! The folder doesn't match its own metadata - unexpected after Pillar 1. Investigate before training")
    print("  (wrong folder pointed at, partially re-copied, or edited since Pillar 1 signed off).")

print("\nComposition (split x class x source):")
print(pd.crosstab([inv["split"], inv["label"]], inv["source"]).to_string())

### 3.1 Decode and cache images
Each image is read once: its MD5 is recorded (for the dataset fingerprint below), and the image is decoded, made upright, converted to RGB and resized to 128x128. The resized images are kept in memory (`RESIZED`) so training never touches the original files again.

In [ ]:
RESIZED = {}          # path -> PIL image (RGB, IMAGE_SIZE); the single source of pixels for the whole notebook

def load_resized(path, size=IMAGE_SIZE):
    """Decode one image: EXIF-upright, RGB, resized to `size`. Returns the image and its MD5."""
    with open(path, "rb") as f:
        data = f.read()
    img = Image.open(io.BytesIO(data))
    img.draft("RGB", size)                          # JPEG only: decode at reduced size, much faster for 12 MP photos
    img.load()
    if APPLY_EXIF_TRANSPOSE:
        img = ImageOps.exif_transpose(img)
    img = img.convert("RGB").resize(size, Image.BICUBIC)
    return img, hashlib.md5(data).hexdigest()

md5s, failures = {}, []
t0 = time.time()
for path in tqdm(inv["path"], desc="Decoding images"):
    try:
        RESIZED[path], md5s[path] = load_resized(path)
    except Exception as e:
        failures.append((path, repr(e)))
print(f"Decoded {len(RESIZED)} images in {time.time() - t0:.0f}s")

if failures:
    print(f"\n{len(failures)} images could not be decoded:")
    for p, e in failures[:10]:
        print("   ", p, e)
    raise RuntimeError("Fix or replace the files above before training.")

inv["md5"] = inv["path"].map(md5s)

# Dataset fingerprint: relative path + label + file hash of every image - lets two computers confirm
# byte-identical data without comparing files directly
fp_lines = sorted(f"{os.path.relpath(r.path, DATASET_ROOT).replace(os.sep, '/')}|{r.label}|{r.md5}" for r in inv.itertuples())
DATASET_FINGERPRINT = hashlib.sha256("\n".join(fp_lines).encode()).hexdigest()[:16]
ENV_INFO["dataset_fingerprint"] = DATASET_FINGERPRINT
ENV_INFO["audit_status"] = AUDIT_STATUS
print(f"Dataset fingerprint: {DATASET_FINGERPRINT}")

inv.drop(columns=["key"]).assign(path=lambda d: [os.path.relpath(p, DATASET_ROOT).replace(os.sep, "/") for p in d["path"]]) \
   .to_csv(f"{DIRS['audit']}/inventory_{COMPUTER_ID}.csv", index=False)
with open(f"{DIRS['audit']}/verification_{COMPUTER_ID}.json", "w") as f:
    json.dump({"status": AUDIT_STATUS, "fingerprint": DATASET_FINGERPRINT, "checks": checks,
              "pillar_1_leakage_audit": {
                  "augmented_copies": "1272/2544", "cross_split_duplicate": 0, "cross_split_augmented": 0,
                  "test_exposed": 0, "validation_exposed": 0, "same_filename_cross_split": 0, "student_public_overlap": 0,
              }}, f, indent=2, default=str)
print(f"Saved verification evidence to {DIRS['audit']}/")

if STOP_ON_AUDIT_FAIL and AUDIT_STATUS == "FAIL":
    raise RuntimeError("Dataset verification failed and STOP_ON_AUDIT_FAIL is True")

## 4. Data Loading & Preprocessing
Preprocessing is defined once and used by all four models:

```
Image file
   ↓
EXIF upright → RGB → resize 128 × 128 (bicubic)      [done once, Section 3.3, cached in RESIZED]
   ↓
ToTensor → float32, pixel values 0–1
   ↓
Model input  (EfficientNet models apply the ImageNet standardisation inside the model)
```

The DataLoaders always deliver 0–1 tensors, so the data pipeline is identical for every model. Pretrained EfficientNet expects ImageNet-standardised input, so that step lives inside the EfficientNet model class (Section 9) rather than in the data pipeline. `num_workers=0` avoids process-spawn problems on Windows; with the images cached in memory the data pipeline is not a bottleneck.

In [ ]:
# Fixed, sorted sample lists per split: (image path, class index)
def samples_for(split):
    d = inv[inv["split"] == split].sort_values("path")
    return list(zip(d["path"], d["label_idx"]))

train_samples, val_samples, test_samples = samples_for("train"), samples_for("validation"), samples_for("test")

if QUICK_TEST:      # Small subset so the whole notebook can be smoke-tested quickly
    train_samples, val_samples, test_samples = train_samples[::8], val_samples[::4], test_samples[::4]

path_to_source = dict(zip(inv["path"], inv["source"]))
test_paths = [p for p, _ in test_samples]
test_labels = np.array([l for _, l in test_samples])
test_sources = np.array([path_to_source[p] for p in test_paths])

class FruitDataset(Dataset):
    """Serves the cached 128x128 images through a transform; returns (tensor, label)."""
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(RESIZED[path]), label

# Evaluation transform: NO augmentation (validation, test and embedding extraction)
eval_transform = transforms.Compose([transforms.ToTensor()])      # PIL image -> float tensor, pixels 0-1

def make_eval_loader(samples):
    return DataLoader(FruitDataset(samples, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

train_eval_loader = make_eval_loader(train_samples)     # Training images without augmentation (used for embeddings)
val_loader = make_eval_loader(val_samples)
test_loader = make_eval_loader(test_samples)

# Check the pipeline output
xb, yb = next(iter(val_loader))
print("=== Preprocessing check ===")
print(f"Batch tensor : {tuple(xb.shape)}  dtype={xb.dtype}  pixel range=[{xb.min():.3f}, {xb.max():.3f}]")
assert xb.shape[1:] == (3, IMAGE_SIZE[0], IMAGE_SIZE[1]) and xb.min() >= 0 and xb.max() <= 1
for name, samples in [("Train", train_samples), ("Validation", val_samples), ("Test", test_samples)]:
    counts = Counter(CLASSES[l] for _, l in samples)
    print(f"{name:<11}: {len(samples):>5} images  {dict(counts)}  -> {math.ceil(len(samples) / BATCH_SIZE)} batches of {BATCH_SIZE}")

#### Reasoning: One Decode, Cached Resize
The source photos are not stored at 128 × 128 (Section 3.3 reports the real on-disk sizes – student phone photos are about 12 megapixels). Decoding and augmenting such files inside the DataLoader every epoch is far slower than the model itself, so every image is decoded **once**, resized to 128 × 128 and kept in memory.

| Decision | Why |
|----------|-----|
| Resize once, then augment the 128 × 128 copy | Augmentation is fast and identical in cost for every model |
| EXIF orientation applied | About 6 in 10 student photos are stored sideways with a rotation tag; without it they would enter the model rotated 90° |
| Plain resize (no crop) | Keeps the whole fruit in view; the small aspect-ratio change of 4:3 photos is the same for every image and every model |
| 0–1 pixels in the data pipeline | Same input for all four models; standardisation for pretrained weights happens inside the model |

## 5. Training Augmentation
Augmentation is applied to the **training data only**. Validation and test images are never augmented in the standard evaluation.

**Split first → augmentation second.** The splits already exist as folders; augmentation happens on the fly, per batch, on the training images only. No augmented copy is ever written to disk or moved between splits.

| Augmentation | Base | Strong | Purpose |
|--------------|------|--------|---------|
| Horizontal flip | p = 0.5 | p = 0.5 | Mirrored fruit is still the same fruit |
| Rotation | ±15° | ±25° | Fruit lying at different angles |
| Zoom (random resized crop, area kept) | 85–100 % | 70–100 % | Different camera distances |
| Brightness / contrast | ±0.2 | ±0.3 | Different lighting |
| Saturation | ±0.1 | ±0.2 | Different cameras and white balance |
| **Not used:** hue shift | – | – | Colour is the strongest class cue (red/green vs yellow); shifting hue could turn an apple yellow |
| **Not used:** vertical flip, noise | – | – | The public images already contain vertically flipped and salt-and-pepper copies (Note: As a stretch goal, the final models is **explored** in Section 18: Robustness under extreme corruption.) |

In [ ]:
AUG_CONFIGS = {
    "base":   {"hflip": 0.5, "rotation": 15, "zoom": (0.85, 1.0), "brightness": 0.2, "contrast": 0.2, "saturation": 0.1},
    "strong": {"hflip": 0.5, "rotation": 25, "zoom": (0.70, 1.0), "brightness": 0.3, "contrast": 0.3, "saturation": 0.2},
}

def build_train_transform(cfg):
    """Training transform: augmentation first, then ToTensor (pixels 0-1)."""
    return transforms.Compose([
        transforms.RandomHorizontalFlip(p=cfg["hflip"]),
        transforms.RandomRotation(cfg["rotation"], interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=cfg["zoom"], ratio=(0.9, 1.1)),
        transforms.ColorJitter(brightness=cfg["brightness"], contrast=cfg["contrast"], saturation=cfg["saturation"], hue=0.0),
        transforms.ToTensor(),
    ])

def make_train_loader(aug_name, seed=SEED):
    """Shuffled training loader with augmentation (the shuffle order is seeded)."""
    ds = FruitDataset(train_samples, build_train_transform(AUG_CONFIGS[aug_name]))
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, generator=g)

# Preview: one original and seven augmented versions of one training image per class
set_seed(SEED)
fig, axes = plt.subplots(2 * len(AUG_CONFIGS), 8, figsize=(16, 4.3 * len(AUG_CONFIGS)))
row = 0
for aug_name, cfg in AUG_CONFIGS.items():
    tf = build_train_transform(cfg)
    for cls_idx in range(len(CLASSES)):
        path = next(p for p, l in train_samples if l == cls_idx)
        for col in range(8):
            img = RESIZED[path] if col == 0 else transforms.ToPILImage()(tf(RESIZED[path]))
            axes[row, col].imshow(img)
            axes[row, col].axis("off")
            axes[row, col].set_title("original" if col == 0 else f"aug {col}", fontsize=8)
        axes[row, 0].text(-0.15, 0.5, f"{aug_name}\n{CLASSES[cls_idx]}", transform=axes[row, 0].transAxes, ha="right", va="center", fontsize=10)
        row += 1
fig.suptitle("Training augmentation preview (training data only)", fontsize=14)
fig.tight_layout(rect=[0.04, 0, 1, 0.96])
save_fig(fig, "audit", f"augmentation_preview_{COMPUTER_ID}")

## 6. Model Configuration
All four experiments are defined **before** any training. Every computer uses exactly these values; a hash of the configuration is stored with every result, so a changed hyperparameter on one computer would be visible immediately in the collation notebook.

#### Reasoning: Why These Four Configurations

| Model | What is different | Question it answers |
|-------|-------------------|---------------------|
| M1 Custom CNN Baseline | Small CNN, random weights, fixed simple setup | What does a from-scratch model achieve? |
| M2 Custom CNN Tuned | Same family, but deeper blocks, more dropout, AdamW + weight decay, cosine LR schedule, stronger augmentation | Does controlled optimisation improve the baseline? (hyperparameter/architecture tuning – *not* transfer-learning fine-tuning) |
| M3 EfficientNetB0 Transfer Learning | Pretrained ImageNet features frozen, only the new head is trained | How good are generic pretrained features for this task? |
| M4 EfficientNetB0 Fine-Tuned | Starts from M3, later stages unfrozen, 10× smaller learning rate | Does adapting pretrained features to this dataset help? |

**Why no label smoothing?** Label smoothing caps the softmax confidence below 100 %, which would distort the confidence and reliability analysis (Sections 15 and 18). All models are trained with plain cross-entropy so the probabilities can be interpreted directly.

**Why is M4 warm-started from M3?** The standard fine-tuning recipe first trains the new head on frozen features (M3) and then unfreezes later layers at a small learning rate. Starting M4 from M3's trained head avoids large, noisy gradients from a randomly initialised head flowing into the pretrained layers.

**M2's values above are defaults.** Section 11.0 (immediately before the 16 experiments actually run) can load a locked M2 configuration that overrides them; the config hash that ends up in every exported result is the one printed there, not the one printed at the end of this section.

In [ ]:
COMMON = {"batch_size": BATCH_SIZE, "min_delta": 1e-3, "label_smoothing": 0.0}

MODEL_CONFIGS = {
    "M1": {**COMMON, "name": "Custom CNN Baseline", "type": "custom_cnn",
           "channels": (32, 64, 128, 256), "convs_per_block": 1, "dropout": 0.25,
           "pretrained": False, "unfreeze_from": None, "init_from": None,
           "optimizer": "adam", "lr": 1e-3, "weight_decay": 0.0, "scheduler": "none",
           "epochs": 30, "patience": 8, "augmentation": "base"},
    "M2": {**COMMON, "name": "Custom CNN Tuned", "type": "custom_cnn",
           "channels": (32, 64, 128, 256), "convs_per_block": 2, "dropout": 0.40,
           "pretrained": False, "unfreeze_from": None, "init_from": None,
           "optimizer": "adamw", "lr": 1e-3, "weight_decay": 1e-2, "scheduler": "cosine",
           "epochs": 40, "patience": 10, "augmentation": "strong"},
    "M3": {**COMMON, "name": "EfficientNetB0 Transfer Learning", "type": "efficientnet_b0",
           "dropout": 0.30, "pretrained": True, "unfreeze_from": None, "init_from": None,
           "optimizer": "adamw", "lr": 1e-3, "weight_decay": 1e-4, "scheduler": "cosine",
           "epochs": 15, "patience": 5, "augmentation": "base"},
    "M4": {**COMMON, "name": "EfficientNetB0 Fine-Tuned", "type": "efficientnet_b0",
           "dropout": 0.30, "pretrained": True, "unfreeze_from": 6, "init_from": "M3",
           "optimizer": "adamw", "lr": 1e-4, "weight_decay": 1e-4, "scheduler": "cosine",
           "epochs": 15, "patience": 5, "augmentation": "base"},
}

if QUICK_TEST:      # Smoke test: 2 epochs per model
    for cfg in MODEL_CONFIGS.values():
        cfg["epochs"], cfg["patience"] = 2, 2

# Configuration hash: proves every computer ran identical settings
CONFIG_SNAPSHOT = {"models": MODEL_CONFIGS, "augmentation": AUG_CONFIGS, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
                   "seed": SEED, "preprocessing": PREPROCESSING_VERSION, "quick_test": QUICK_TEST}
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG_SNAPSHOT, sort_keys=True, default=str).encode()).hexdigest()[:12]
ENV_INFO["config_hash"] = CONFIG_HASH

# Configuration table
cfg_table = pd.DataFrame({
    "Model": [f"{m} - {MODEL_CONFIGS[m]['name']}" for m in MODEL_IDS],
    "Optimizer": [MODEL_CONFIGS[m]["optimizer"] for m in MODEL_IDS],
    "LR": [MODEL_CONFIGS[m]["lr"] for m in MODEL_IDS],
    "Weight decay": [MODEL_CONFIGS[m]["weight_decay"] for m in MODEL_IDS],
    "Dropout": [MODEL_CONFIGS[m]["dropout"] for m in MODEL_IDS],
    "Scheduler": [MODEL_CONFIGS[m]["scheduler"] for m in MODEL_IDS],
    "Max epochs": [MODEL_CONFIGS[m]["epochs"] for m in MODEL_IDS],
    "Patience": [MODEL_CONFIGS[m]["patience"] for m in MODEL_IDS],
    "Augmentation": [MODEL_CONFIGS[m]["augmentation"] for m in MODEL_IDS],
})
print("=" * 80)
print(f"MODEL CONFIGURATION (batch size {BATCH_SIZE}, seed {SEED}, config hash {CONFIG_HASH})")
print("=" * 80)
print(cfg_table.to_string(index=False))

# Experiment IDs: Model ID + Computer ID
ALL_EXPERIMENT_IDS = [f"{m}_{pc}" for m in MODEL_IDS for pc in COMPUTER_IDS]
print(f"\nExperiment IDs ({len(ALL_EXPERIMENT_IDS)} runs in total):")
for m in MODEL_IDS:
    print("  " + "   ".join(f"{m}_{pc}" + (" <- this computer" if pc == COMPUTER_ID else "") for pc in COMPUTER_IDS))

## 7. Model 1 – Custom CNN Baseline
A small CNN, **randomly initialised** and trained from scratch, with a fixed, simple configuration. This is the baseline every other model is compared against. Model 2 (Section 8) reuses the same `CustomCNN` class with different hyperparameters, so the architecture is defined once here.

In [ ]:
class CustomCNN(nn.Module):
    """Conv-BN-ReLU blocks (one per entry in `channels`), global average pooling, dropout + linear head.
    `convs_per_block` repeats the Conv-BN-ReLU inside a block before downsampling (M1 uses 1, M2 uses 2)."""
    def __init__(self, channels=(32, 64, 128, 256), convs_per_block=1, dropout=0.25, num_classes=2):
        super().__init__()
        blocks, in_ch = [], 3
        for out_ch in channels:
            layers = []
            for i in range(convs_per_block):
                layers += [nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1),
                          nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
            layers.append(nn.MaxPool2d(2))
            blocks.append(nn.Sequential(*layers))
            in_ch = out_ch
        self.features = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_ch, num_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

def build_custom_cnn(cfg):
    return CustomCNN(channels=cfg["channels"], convs_per_block=cfg["convs_per_block"],
                     dropout=cfg["dropout"], num_classes=len(CLASSES))

# Preview M1
_preview = build_custom_cnn(MODEL_CONFIGS["M1"])
n_params = sum(p.numel() for p in _preview.parameters())
print(f"M1 Custom CNN Baseline: {n_params:,} parameters, {len(MODEL_CONFIGS['M1']['channels'])} conv blocks "
      f"{MODEL_CONFIGS['M1']['channels']}, {MODEL_CONFIGS['M1']['convs_per_block']} conv/block, "
      f"dropout {MODEL_CONFIGS['M1']['dropout']}")
del _preview

#### Reasoning: Baseline Design
The baseline is deliberately simple: one convolution per block, moderate dropout (0.25), plain Adam, no learning-rate schedule, no augmentation beyond the base set. Apple vs banana is a two-class, colour-and-shape task, so a small model (channel widths 32→64→128→256) has enough capacity without needing residual connections. Keeping M1 simple is intentional: it is the reference that Section 8 asks "does *controlled* optimisation improve on this?" against.

## 8. Model 2 – Custom CNN Tuned
Same `CustomCNN` architecture family as Model 1, with **controlled optimisation**: two convolutions per block (more capacity per block instead of a wider/deeper network), higher dropout, AdamW with weight decay, a cosine learning-rate schedule, and the stronger augmentation set.

**This is hyperparameter/architecture tuning, not transfer-learning fine-tuning** – there are no pretrained weights involved anywhere in M1 or M2.

In [ ]:
_m1, _m2 = MODEL_CONFIGS["M1"], MODEL_CONFIGS["M2"]
diff_table = pd.DataFrame({
    "Setting": ["Convs per block", "Dropout", "Optimizer", "Learning rate", "Weight decay", "Scheduler", "Augmentation", "Max epochs"],
    "M1 Baseline": [_m1["convs_per_block"], _m1["dropout"], _m1["optimizer"], _m1["lr"], _m1["weight_decay"], _m1["scheduler"], _m1["augmentation"], _m1["epochs"]],
    "M2 Tuned":    [_m2["convs_per_block"], _m2["dropout"], _m2["optimizer"], _m2["lr"], _m2["weight_decay"], _m2["scheduler"], _m2["augmentation"], _m2["epochs"]],
})
print("M1 vs M2 configuration differences:")
print(diff_table.to_string(index=False))

_preview = build_custom_cnn(MODEL_CONFIGS["M2"])
print(f"\nM2 Custom CNN Tuned: {sum(p.numel() for p in _preview.parameters()):,} parameters")
del _preview

#### Reasoning: What "Tuned" Means Here
Two convolutions per block roughly doubles the model's representational capacity per spatial resolution, which is why dropout is raised from 0.25 to 0.40 and weight decay is added (AdamW, 1e-2) – more capacity needs more regularisation on ~1,780 training images. The cosine schedule anneals the learning rate smoothly across the whole training run instead of holding it fixed, which is more forgiving than a step decay for a run whose optimal number of epochs isn't known in advance (early stopping decides that per computer). The stronger augmentation set compensates for the extra capacity by making the training data harder to memorise.

A validation-only hyperparameter sweep that produced these choices is available in Section 11.0 (`RUN_TUNING_SWEEP`), off by default – it is for the PC1 who prepared the master notebook, not something each of the other three teammates should turn on, since the whole point of the 16-run design is that everyone trains the *same* M2.

## 9. Model 3 – EfficientNetB0 Transfer Learning
```
Pretrained EfficientNetB0
        ↓
     Frozen
        ↓
Classification Head
        ↓
Apple / Banana
```
The pretrained backbone is frozen; only a new classification head is trained. ImageNet standardisation (mean/std) is applied **inside the model**, not in the data pipeline, so every model in this notebook (Sections 7–10) receives exactly the same 0–1 tensors from the DataLoader.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class EfficientNetWrapper(nn.Module):
    """EfficientNetB0 with a new 2-class head. Input is a raw 0-1 tensor; ImageNet mean/std
    standardisation happens inside forward(). `unfreeze_from=None` freezes the whole backbone (M3);
    an integer unfreezes backbone blocks from that index onwards, earlier blocks stay frozen (M4)."""
    def __init__(self, dropout=0.30, unfreeze_from=None, num_classes=2, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.efficientnet_b0(weights=weights)
        self.features = backbone.features
        self.pool = backbone.avgpool
        in_features = backbone.classifier[1].in_features
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

        for p in self.features.parameters():          # Start fully frozen ...
            p.requires_grad = False
        if unfreeze_from is not None:                  # ... then unfreeze later blocks for fine-tuning
            for block in self.features[unfreeze_from:]:
                for p in block.parameters():
                    p.requires_grad = True

    def forward(self, x):
        x = (x - self.mean) / self.std
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

def build_efficientnet(cfg, pretrained=True):
    return EfficientNetWrapper(dropout=cfg["dropout"], unfreeze_from=cfg["unfreeze_from"],
                               num_classes=len(CLASSES), pretrained=pretrained)

def build_model(model_id, pretrained=True):
    """Dispatch to the right builder for any of M1-M4."""
    cfg = MODEL_CONFIGS[model_id]
    if cfg["type"] == "custom_cnn":
        return build_custom_cnn(cfg)
    return build_efficientnet(cfg, pretrained=pretrained)

### 9.1 Fetch the pretrained weights now
Doing this once, with a timeout, means a blocked network shows up here as a clear error instead of a silent freeze later during training.

In [ ]:
_weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
print("EfficientNet-B0 ImageNet weights:", _weights.url)
_old_timeout = socket.getdefaulttimeout()
socket.setdefaulttimeout(60)
try:
    _probe = models.efficientnet_b0(weights=_weights)
    print("OK - weights available (downloaded now or already cached).")
    del _probe
except Exception as e:
    print("Could not fetch the pretrained weights:", repr(e))
    print("Fix: check the internet connection/proxy, or download the file from the URL above into:")
    print("   ", os.path.join(torch.hub.get_dir(), "checkpoints"))
    raise
finally:
    socket.setdefaulttimeout(_old_timeout)

_preview = build_model("M3")
n_total = sum(p.numel() for p in _preview.parameters())
n_trainable = sum(p.numel() for p in _preview.parameters() if p.requires_grad)
print(f"\nM3 EfficientNetB0 Transfer Learning: {n_total:,} total parameters, {n_trainable:,} trainable (the new head)")
del _preview

#### Reasoning: Why Freeze the Whole Backbone for M3
Freezing the backbone turns M3 into a linear probe on top of ImageNet features. This is a deliberately simple baseline for transfer learning: ImageNet-1k actually contains "banana" and "Granny Smith" as real classes, so the frozen features are already tuned for exactly these fruits, and near-ceiling accuracy from a linear head is an expected result of transfer learning working, not a sign of a bug. Section 4b's speed check estimates training time before this model runs.

## 10. Model 4 – EfficientNetB0 Fine-Tuned
```
Early layers       → Frozen
Later layers       → Trainable
Classification head → Trainable
```
Starts from the transfer-learning configuration (Model 3) and unfreezes the later pretrained layers, with a smaller learning rate. The warm start from M3's trained head happens in Section 11, where M3 is guaranteed to have finished training first.

In [ ]:
_preview = build_model("M4")
n_total = sum(p.numel() for p in _preview.parameters())
n_trainable = sum(p.numel() for p in _preview.parameters() if p.requires_grad)
n_blocks = len(_preview.features)
print(f"M4 EfficientNetB0 Fine-Tuned: unfreezing backbone blocks {MODEL_CONFIGS['M4']['unfreeze_from']}-{n_blocks - 1} "
      f"of {n_blocks} (0 = stem, {n_blocks - 1} = final 1x1 conv)")
print(f"{n_total:,} total parameters, {n_trainable:,} trainable "
      f"(compare to M3's {sum(p.numel() for p in build_model('M3', pretrained=False).parameters() if p.requires_grad):,})")
del _preview

#### Reasoning: Warm Start and a Smaller Learning Rate
Starting M4 from a randomly initialised head would send large, noisy gradients back into the newly unfrozen pretrained layers and could damage useful pretrained features before the head has learned anything. Warm-starting from M3's already-trained head avoids that, and using a learning rate 10× smaller than M3's (1e-4 vs 1e-3) protects the pretrained weights from being overwritten too aggressively (the standard fine-tuning recipe). Only the last third of the backbone is unfrozen: those layers hold the more task-specific features, while the early layers (edges, colour blobs, textures) are already generic and useful as they are.

**Why does this matter for the confidence gate?** M3's frozen head can be a bit under-confident on genuinely ambiguous images since it never adapts its features to *this* dataset. Fine-tuning is the natural next step to test whether adapting the features improves both accuracy and the fraction of predictions that clear the 0.90 confidence threshold (Section 15).

## 11. Training & Experiment Logging
One training engine is shared by all four models. For every model: build → train (with early stopping on the **validation set**, Section 1's fixed split – there is no cross-validation in this design) → save history, checkpoint and metadata → evaluate once on validation for logging (the **test set is only touched in Section 12**, after every model has finished training).

If a model already has a saved result for this computer (`RESUME = True`), it is loaded instead of retrained – safe to use after an interrupted run.

In [ ]:
def set_train_mode(model):
    """model.train(), but a BatchNorm layer whose own parameters are all frozen is kept in eval mode,
    so it stops updating its running statistics. Needed for M3 (fully frozen) and M4 (partly frozen) -
    without this, a frozen backbone's pretrained BatchNorm statistics get overwritten during training."""
    model.train()
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d) and not any(p.requires_grad for p in module.parameters()):
            module.eval()

def build_optimizer(model, cfg):
    params = [p for p in model.parameters() if p.requires_grad]
    if cfg["optimizer"] == "adamw":
        return optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    return optim.Adam(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])

def build_scheduler(optimizer, cfg):
    if cfg["scheduler"] == "cosine":
        return lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"], eta_min=cfg["lr"] * 0.01)
    return None

def train_one_epoch(model, loader, optimizer, criterion, log_every=0, tag=""):
    set_train_mode(model)
    total, correct, loss_sum = 0, 0, 0.0
    t0 = time.time()
    for b, (x, y) in enumerate(loader, 1):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
        if log_every and b % log_every == 0:
            print(f"      {tag} batch {b}/{len(loader)}  ({time.time() - t0:.0f}s)", flush=True)
    return loss_sum / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    all_probs, all_preds, all_labels = [], [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        if criterion is not None:
            loss_sum += criterion(logits, y).item() * y.size(0)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        all_probs.append(probs.cpu()); all_preds.append(preds.cpu()); all_labels.append(y.cpu())
    return {"loss": loss_sum / total if criterion else None, "acc": correct / total,
            "probs": torch.cat(all_probs).numpy(), "preds": torch.cat(all_preds).numpy(),
            "labels": torch.cat(all_labels).numpy()}

def fit(model, train_loader, val_loader, cfg, ckpt_path, tag=""):
    """Train with early stopping on validation loss. Restores and returns the best epoch's weights."""
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg)

    best_loss, best_state, wait = float("inf"), None, 0
    best_epoch, history = 0, {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, cfg["epochs"] + 1):
        t_ep = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion,
                                          log_every=20 if epoch == 1 else 0, tag=tag)
        va = evaluate(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()
        history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
        history["val_loss"].append(va["loss"]); history["val_acc"].append(va["acc"])

        improved = va["loss"] < best_loss - cfg["min_delta"]
        if improved:
            best_loss, best_state, best_epoch, wait = va["loss"], copy.deepcopy(model.state_dict()), epoch, 0
        else:
            wait += 1

        print(f"  {tag} ep {epoch:02d}/{cfg['epochs']} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
              f"val loss {va['loss']:.3f} acc {va['acc']:.3f} | {time.time() - t_ep:.0f}s | "
              f"best@{best_epoch} wait {wait}/{cfg['patience']}", flush=True)

        if wait >= cfg["patience"]:
            print(f"  {tag} early stopping at epoch {epoch} (keeping epoch {best_epoch})", flush=True)
            break

    model.load_state_dict(best_state if best_state is not None else model.state_dict())
    os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
    torch.save(model.state_dict(), ckpt_path)
    return model, {"best_epoch": best_epoch, "epochs_run": epoch, "best_val_loss": best_loss}, history

### 11.0 Lock Model 2's hyperparameters (run once, before any of the 16 experiments)
`sweep → lock → train M2 → run 16 experiments`, in that order. The master notebook (PC1) runs this cell once with `RUN_TUNING_SWEEP = True`; it searches Section 8's settings, writes the winner to `M2_locked_config.json` (in the same folder as this notebook, not inside `results/`), and that file is then sent to PC2, PC3 and PC4 **alongside** the notebook. Every computer's own run of this cell (with `RUN_TUNING_SWEEP = False`, the default) loads that same file and overrides `MODEL_CONFIGS["M2"]` with it *before* Section 11.1's training loop runs below, so M2 always trains with the locked values on every computer. If no lock file exists yet, M2 keeps the Section 6 defaults and a warning is printed.

The configuration hash is recomputed at the end of this cell, once M2's final values are known - this is the hash that appears in every experiment log, not the one printed in Section 6.

In [ ]:
M2_LOCK_PATH = "M2_locked_config.json"     # Lives next to this notebook and include it when distributing the master notebook

def apply_m2_lock(path=M2_LOCK_PATH):
    """If a locked M2 configuration exists, overwrite MODEL_CONFIGS['M2']'s tunable keys with it and
    return the lock record; otherwise return None (M2 keeps its Section 6 defaults)."""
    if not os.path.exists(path):
        return None
    with open(path) as f:
        lock = json.load(f)
    MODEL_CONFIGS["M2"].update(lock["chosen"])
    return lock

if RUN_TUNING_SWEEP:
    if os.path.exists(M2_LOCK_PATH) and not OVERWRITE_LOCK:
        raise RuntimeError(f"'{M2_LOCK_PATH}' already exists - re-running the sweep would silently change what "
                           f"PC2/PC3/PC4 train if they haven't pulled the new file yet. Set OVERWRITE_LOCK = True "
                           f"in Section 2.2 to intentionally re-lock.")
    sweep_space = [
        {"dropout": 0.30, "lr": 1e-3, "weight_decay": 1e-2},
        {"dropout": 0.40, "lr": 1e-3, "weight_decay": 1e-2},
        {"dropout": 0.40, "lr": 3e-4, "weight_decay": 1e-2},
        {"dropout": 0.50, "lr": 1e-3, "weight_decay": 5e-2},
    ]
    sweep_rows = []
    for i, overrides in enumerate(sweep_space):
        cfg = {**MODEL_CONFIGS["M2"], **overrides, "epochs": 8, "patience": 3}
        set_seed(SEED)
        model = build_custom_cnn(cfg).to(device)
        train_loader = make_train_loader(cfg["augmentation"])
        _, fit_info, _ = fit(model, train_loader, val_loader, cfg, f"{DIRS['checkpoints']}/_sweep_{i}.pt", tag=f"[sweep {i}]")
        sweep_rows.append({**overrides, "best_val_loss": fit_info["best_val_loss"], "epochs_run": fit_info["epochs_run"]})
    sweep_df = pd.DataFrame(sweep_rows).sort_values("best_val_loss")
    print(sweep_df.to_string(index=False))
    winner = sweep_df.iloc[0][["dropout", "lr", "weight_decay"]].to_dict()
    print(f"\nBest by validation loss: {winner}")

    lock_record = {"chosen": winner, "locked_by_computer": COMPUTER_ID,
                  "locked_at": datetime.datetime.now().isoformat(timespec="seconds"),
                  "dataset_fingerprint": DATASET_FINGERPRINT, "notebook_version": NOTEBOOK_VERSION,
                  "sweep_candidates": sweep_rows}
    with open(M2_LOCK_PATH, "w") as f:
        json.dump(lock_record, f, indent=2, default=str)
    sweep_df.to_csv(f"{DIRS['experiment_logs']}/M2_tuning_sweep_{COMPUTER_ID}.csv", index=False)
    print(f"\nLocked to '{M2_LOCK_PATH}' - send this file to PC2, PC3 and PC4 along with the notebook, before any of them run Section 11.1.")
else:
    print("RUN_TUNING_SWEEP is off - checking for an existing lock file instead.")

m2_lock = apply_m2_lock()
if m2_lock:
    print(f"\nLoaded locked M2 hyperparameters from '{M2_LOCK_PATH}':")
    print(f"  {m2_lock['chosen']}  (locked by {m2_lock['locked_by_computer']} at {m2_lock['locked_at']})")
    if m2_lock.get("dataset_fingerprint") != DATASET_FINGERPRINT:
        print(f"  WARNING!!! that lock's dataset_fingerprint doesn't match this computer's ({DATASET_FINGERPRINT}) - "
              f"the sweep may have run on different data. Check before trusting the 16-run comparison.")
else:
    print(f"\nNo '{M2_LOCK_PATH}' found next to this notebook - M2 will train with the Section 6 defaults.")
    print("For a fully controlled 16-run experiment, run this cell once with RUN_TUNING_SWEEP = True, then send the")
    print(f"resulting '{M2_LOCK_PATH}' to every computer before treating any of the 16 runs as final.")

# Recompute the configuration hash now that M2's final values are known - this is the hash that goes
# into every experiment log below, not the (possibly pre-lock) one printed at the end of Section 6.
CONFIG_SNAPSHOT = {"models": MODEL_CONFIGS, "augmentation": AUG_CONFIGS, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
                   "seed": SEED, "preprocessing": PREPROCESSING_VERSION, "quick_test": QUICK_TEST}
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG_SNAPSHOT, sort_keys=True, default=str).encode()).hexdigest()[:12]
ENV_INFO["config_hash"] = CONFIG_HASH
print(f"\nFinal M2 configuration: dropout={MODEL_CONFIGS['M2']['dropout']}  lr={MODEL_CONFIGS['M2']['lr']}  "
     f"weight_decay={MODEL_CONFIGS['M2']['weight_decay']}")
print(f"Config hash used for training below: {CONFIG_HASH}")

with open(f"{DIRS['experiment_logs']}/M2_config_used_{COMPUTER_ID}.json", "w") as f:
    json.dump({"model_config": MODEL_CONFIGS["M2"], "lock_loaded": m2_lock is not None, "config_hash": CONFIG_HASH},
              f, indent=2, default=str)

### 11.1 – 11.4 Run all four experiments
For every model: load dataset → preprocess → apply training augmentation → build model → train → validate → save history → save checkpoint → evaluate → export results (matching Section 9's "Training Outputs" list exactly).

In [ ]:
RESULTS = {}          # model_id -> metadata dict (also written to disk)
HISTORIES = {}         # model_id -> training history dict
TRAINED_MODELS = {}    # model_id -> trained model, kept in memory for Sections 12-19

def run_experiment(model_id):
    cfg = MODEL_CONFIGS[model_id]
    eid = exp_id(model_id)
    ckpt_path = f"{DIRS['checkpoints']}/{eid}.pt"
    meta_path = f"{DIRS['experiment_logs']}/{eid}.json"

    if RESUME and os.path.exists(meta_path) and os.path.exists(ckpt_path):
        with open(meta_path) as f:
            meta = json.load(f)
        model = build_model(model_id, pretrained=False)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        TRAINED_MODELS[model_id] = model.to(device)
        RESULTS[model_id] = meta
        HISTORIES[model_id] = meta["history"]
        print(f"{eid}: already finished -> loaded from {meta_path}")
        return

    print(f"\n{'=' * 60}\n{eid}  -  {cfg['name']}\n{'=' * 60}", flush=True)
    set_seed(SEED)

    model = build_model(model_id).to(device)
    if cfg["init_from"] is not None:                        # M4 warm-starts from M3's trained head
        src_ckpt = f"{DIRS['checkpoints']}/{exp_id(cfg['init_from'])}.pt"
        model.load_state_dict(torch.load(src_ckpt, map_location=device))
        # Re-apply the freeze mask: loading the state dict does not change requires_grad
        for p in model.features.parameters():
            p.requires_grad = False
        for block in model.features[cfg["unfreeze_from"]:]:
            for p in block.parameters():
                p.requires_grad = True
        print(f"  warm-started from {cfg['init_from']} ({src_ckpt})")

    train_loader = make_train_loader(cfg["augmentation"])
    t0 = time.time()
    model, fit_info, history = fit(model, train_loader, val_loader, cfg, ckpt_path, tag=f"[{eid}]")
    train_time_s = time.time() - t0

    val_res = evaluate(model, val_loader, nn.CrossEntropyLoss())
    n_params_total = sum(p.numel() for p in model.parameters())
    n_params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    meta = {
        "experiment_id": eid, "model_id": model_id, "model_name": cfg["name"], "computer_id": COMPUTER_ID,
        "dataset_version": DATASET_VERSION, "preprocessing_version": PREPROCESSING_VERSION,
        "notebook_version": NOTEBOOK_VERSION, "config_hash": CONFIG_HASH, "dataset_fingerprint": DATASET_FINGERPRINT,
        "hyperparameters": cfg, "checkpoint_path": ckpt_path,
        "epochs": fit_info["epochs_run"], "best_epoch": fit_info["best_epoch"],
        "train_time_s": train_time_s, "n_params_total": n_params_total, "n_params_trainable": n_params_trainable,
        "val_loss": val_res["loss"], "val_acc": val_res["acc"], "history": history, "environment": ENV_INFO,
    }
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2, default=str)

    RESULTS[model_id] = meta
    HISTORIES[model_id] = history
    TRAINED_MODELS[model_id] = model
    print(f"{eid}: val_acc={val_res['acc']:.4f}  val_loss={val_res['loss']:.4f}  "
          f"epochs={fit_info['epochs_run']} (best {fit_info['best_epoch']})  time={train_time_s / 60:.1f} min", flush=True)

for model_id in MODEL_IDS:
    run_experiment(model_id)

print(f"\nAll {len(MODEL_IDS)} models finished on {COMPUTER_ID}.")

## 12. Standard Model Evaluation
Every model is evaluated on the **same held-out test set** (382 images, untouched until now) using accuracy, precision, recall and F1 (macro-averaged over Apple/Banana), plus the full confusion matrix.

In [ ]:
TEST_PREDS = {}     # model_id -> evaluate() result dict on the test set, reused by later sections

def test_predict(model_id):
    if model_id not in TEST_PREDS:
        TEST_PREDS[model_id] = evaluate(TRAINED_MODELS[model_id], test_loader, nn.CrossEntropyLoss())
    return TEST_PREDS[model_id]

metric_rows = []
for model_id in MODEL_IDS:
    res = test_predict(model_id)
    precision, recall, f1, _ = precision_recall_fscore_support(res["labels"], res["preds"], average="macro", zero_division=0)
    metric_rows.append({"Model": model_id, "Name": MODEL_CONFIGS[model_id]["name"], "Accuracy": res["acc"],
                        "Precision": precision, "Recall": recall, "F1": f1, "Test loss": res["loss"]})

metrics_df = pd.DataFrame(metric_rows)
print("=" * 70)
print(f"TEST SET METRICS ({COMPUTER_ID}, n={len(test_labels)})")
print("=" * 70)
print(metrics_df.assign(**{c: metrics_df[c].map("{:.4f}".format) for c in ["Accuracy", "Precision", "Recall", "F1", "Test loss"]}).to_string(index=False))
metrics_df.to_csv(f"{DIRS['metrics']}/test_metrics_{COMPUTER_ID}.csv", index=False)
for model_id in MODEL_IDS:
    pd.DataFrame([metrics_df[metrics_df["Model"] == model_id].iloc[0]]).to_csv(f"{DIRS['metrics']}/{exp_id(model_id)}.csv", index=False)

### 12.1 Confusion matrices
Apple → Apple, Apple → Banana, Banana → Apple, Banana → Banana for each model.

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_IDS), figsize=(4.2 * len(MODEL_IDS), 4))
for ax, model_id in zip(axes, MODEL_IDS):
    res = TEST_PREDS[model_id]
    cm = confusion_matrix(res["labels"], res["preds"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, xticklabels=CLASSES, yticklabels=CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{model_id} - {MODEL_CONFIGS[model_id]['name']}", fontsize=10)
fig.suptitle(f"Confusion matrices on the test set ({COMPUTER_ID})", y=1.04, fontsize=13)
fig.tight_layout()
save_fig(fig, "confusion_matrices", f"confusion_matrices_{COMPUTER_ID}")

### 12.2 Learning curves
Training/validation accuracy and loss vs epoch, for identifying overfitting, underfitting, convergence and instability.

In [ ]:
fig, axes = plt.subplots(2, len(MODEL_IDS), figsize=(4.2 * len(MODEL_IDS), 7), sharex=False)
for col, model_id in enumerate(MODEL_IDS):
    h = HISTORIES[model_id]
    epochs = np.arange(1, len(h["train_loss"]) + 1)
    axes[0, col].plot(epochs, h["train_acc"], label="train"); axes[0, col].plot(epochs, h["val_acc"], label="val")
    axes[0, col].set_title(f"{model_id} accuracy", fontsize=10); axes[0, col].set_xlabel("epoch"); axes[0, col].legend(fontsize=8)
    axes[1, col].plot(epochs, h["train_loss"], label="train"); axes[1, col].plot(epochs, h["val_loss"], label="val")
    axes[1, col].set_title(f"{model_id} loss", fontsize=10); axes[1, col].set_xlabel("epoch"); axes[1, col].legend(fontsize=8)
fig.suptitle(f"Learning curves ({COMPUTER_ID})", y=1.02, fontsize=13)
fig.tight_layout()
save_fig(fig, "learning_curves", f"learning_curves_{COMPUTER_ID}")

for model_id in MODEL_IDS:
    pd.DataFrame(HISTORIES[model_id]).to_csv(f"{DIRS['learning_curves']}/{exp_id(model_id)}_history.csv", index_label="epoch_idx")

### 12.3 Student vs Public generalisation
The dataset is deliberately 50/50 Student/Public. Evaluated separately here to see whether performance changes across image sources – compared descriptively, not as a pass/fail check.

In [ ]:
source_rows = []
for model_id in MODEL_IDS:
    res = TEST_PREDS[model_id]
    for src in ["Student", "Public"]:
        mask = test_sources == src
        precision, recall, f1, _ = precision_recall_fscore_support(res["labels"][mask], res["preds"][mask], average="macro", zero_division=0)
        source_rows.append({"Model": model_id, "Source": src, "n": int(mask.sum()),
                            "Accuracy": float((res["preds"][mask] == res["labels"][mask]).mean()),
                            "Precision": precision, "Recall": recall, "F1": f1})
source_df = pd.DataFrame(source_rows)
pivot = source_df.pivot(index="Model", columns="Source", values="Accuracy")
pivot["gap"] = (pivot["Student"] - pivot["Public"]).abs()
print("Accuracy by source:")
print(pivot.round(4).to_string())
if (pivot["gap"] > 0.05).any():
    print("\nWARNING!!! Student/Public accuracy gap > 5% for:", list(pivot[pivot["gap"] > 0.05].index))
source_df.to_csv(f"{DIRS['metrics']}/student_vs_public_{COMPUTER_ID}.csv", index=False)

#### Reasoning: Macro-Averaged Metrics, One Test Pass
Precision/recall/F1 are macro-averaged over Apple and Banana (the simple mean of the per-class score) rather than "positive-class" binary metrics, because there is no natural positive class here – both fruits matter equally, and macro-averaging keeps the two classes weighted equally even though the test split is 192/190 rather than perfectly even. Every model calls `test_predict()` at most once; the result is cached in `TEST_PREDS` and reused by every later section (error analysis, confidence, feature space, TTA) so the test set is never accidentally re-evaluated with a different model state.

## 13. Cross-Computer Reproducibility
This is one of the distinctive parts of the project: the same notebook, code and configuration, run on four different computers. This section builds the Model × Computer table from whatever `experiment_logs/` and `metrics/` files are present in this results folder – just this computer's own row on a first run, or the full 4×4 table once all four computers' results have been copied together (e.g. inside `Results_Collation_and_Comparison.ipynb`).

**We are testing reproducibility/consistency across environments, not claiming that four computers make the model more robust.** Differences between computers can appear because the hardware/software environments differ – that is exactly what this table is for investigating, not a problem to hide.

In [ ]:
def load_all_experiment_logs():
    logs = {}
    for path in sorted(glob.glob(f"{DIRS['experiment_logs']}/M*_PC*.json")):
        eid = os.path.basename(path)[:-5]
        if eid.count("_") == 1 and eid.split("_")[0] in MODEL_IDS and eid.split("_")[1] in COMPUTER_IDS:
            with open(path) as f:
                logs[eid] = json.load(f)
    return logs

def load_all_test_metrics():
    rows = {}
    for path in sorted(glob.glob(f"{DIRS['metrics']}/M*_PC*.csv")):
        eid = os.path.basename(path)[:-4]
        if eid.count("_") == 1 and eid.split("_")[0] in MODEL_IDS and eid.split("_")[1] in COMPUTER_IDS:
            rows[eid] = pd.read_csv(path).iloc[0]
    return rows

ALL_LOGS = load_all_experiment_logs()
ALL_TEST_METRICS = load_all_test_metrics()
computers_present = sorted({eid.split("_")[1] for eid in ALL_LOGS})
print(f"Experiment logs found for computer(s): {computers_present}  ({len(ALL_LOGS)} of {len(MODEL_IDS) * len(COMPUTER_IDS)} possible runs)")

def cross_computer_table(get_value, logs_or_metrics):
    rows = {}
    for eid, item in logs_or_metrics.items():
        m, pc = eid.split("_")
        rows.setdefault(m, {})[pc] = get_value(item)
    return pd.DataFrame(rows).T.reindex(index=MODEL_IDS, columns=COMPUTER_IDS)

acc_table  = cross_computer_table(lambda r: r["Accuracy"], ALL_TEST_METRICS)
f1_table   = cross_computer_table(lambda r: r["F1"], ALL_TEST_METRICS)
time_table = cross_computer_table(lambda r: r["train_time_s"], ALL_LOGS)
conv_table = cross_computer_table(lambda r: f"{r['best_epoch']}/{r['epochs']}", ALL_LOGS)

print("\nTest accuracy by model x computer:");  print(acc_table.round(4).to_string())
print("\nTest F1 by model x computer:");        print(f1_table.round(4).to_string())
print("\nTraining time (s) by model x computer:"); print(time_table.round(0).to_string())
print("\nConvergence (best epoch / epochs run):");  print(conv_table.to_string())

if len(computers_present) >= 2:
    spread = acc_table.max(axis=1) - acc_table.min(axis=1)
    print(f"\nAccuracy spread across computers (max - min) per model:\n{spread.round(4).to_string()}")
    if (spread > 0.03).any():
        print("WARNING!!! Spread > 3% for:", list(spread[spread > 0.03].index), "- worth comparing the environment tables (Section 2.4).")
else:
    print(f"\nOnly {computers_present} present so far. The full 4-computer table fills in once every computer's")
    print("results/ folder (or just experiment_logs/, metrics/) has been copied into this one.")

for name, table in [("accuracy", acc_table), ("f1", f1_table), ("train_time_s", time_table), ("convergence", conv_table)]:
    table.to_csv(f"{DIRS['reproducibility']}/{name}_by_model_and_computer.csv")

# This computer's own environment row, for comparing against the other three
env_row = pd.DataFrame([{"Computer": COMPUTER_ID, "OS": ENV_INFO["os"], "PyTorch": ENV_INFO["pytorch"],
                         "Device": ENV_INFO["device_used"], "CPU": ENV_INFO["cpu"]["model"], "RAM (GB)": ENV_INFO["ram_gb"],
                         "Threads used": ENV_INFO["cpu_threads_used"], "Dataset fingerprint": DATASET_FINGERPRINT,
                         "Config hash": CONFIG_HASH}])
env_row.to_csv(f"{DIRS['reproducibility']}/environment_{COMPUTER_ID}.csv", index=False)
print(f"\nSaved this computer's environment row to {DIRS['reproducibility']}/environment_{COMPUTER_ID}.csv")

#### Reasoning: Fingerprints Before Numbers
Before comparing accuracy across computers it is worth checking that `dataset_fingerprint` and `config_hash` (Section 3.5 and Section 6) match across every experiment log. If they don't, an accuracy difference could simply mean two computers trained on different data or different settings – not a reproducibility finding. The collation notebook should check this first.

## 14. Error Analysis
For every incorrect test prediction: image, true class, predicted class, confidence, source and the experiment/model that made the mistake.

In [ ]:
def error_table(model_id):
    res = TEST_PREDS[model_id]
    conf = res["probs"][np.arange(len(res["preds"])), res["preds"]]
    wrong = res["preds"] != res["labels"]
    return pd.DataFrame({
        "experiment_id": exp_id(model_id), "filename": [os.path.basename(p) for p in np.array(test_paths)[wrong]],
        "true": [CLASSES[l] for l in res["labels"][wrong]], "predicted": [CLASSES[p] for p in res["preds"][wrong]],
        "confidence": conf[wrong], "source": test_sources[wrong],
    })

ERROR_TABLES = {m: error_table(m) for m in MODEL_IDS}
for m, df in ERROR_TABLES.items():
    print(f"{m}: {len(df)}/{len(test_labels)} misclassified ({len(df) / len(test_labels):.1%})", end="")
    if len(df):
        print(f"  |  by source: {dict(df['source'].value_counts())}  |  mean confidence on wrong predictions: {df['confidence'].mean():.3f}")
    else:
        print()
    df.to_csv(f"{DIRS['error_analysis']}/{exp_id(m)}_errors.csv", index=False)

### 14.1 Representative misclassified images
Shown for manual review – look for patterns involving lighting, viewpoint, occlusion, background, image quality, unusual fruit appearance or genuinely ambiguous examples. **This step needs a human eye**: the notebook cannot label "poor lighting" on its own, it can only surface the clearest candidates (highest-confidence mistakes first, since a confident wrong answer is the most informative kind of error).

In [ ]:
worst_model = max(ERROR_TABLES, key=lambda m: len(ERROR_TABLES[m]))
show_df = ERROR_TABLES[worst_model].sort_values("confidence", ascending=False).head(12)
path_by_name = {os.path.basename(p): p for p in test_paths}

if len(show_df):
    cols = min(4, len(show_df)); rows_ = math.ceil(len(show_df) / cols)
    fig, axes = plt.subplots(rows_, cols, figsize=(3.3 * cols, 3.4 * rows_))
    axes = np.atleast_1d(axes).flatten()
    for ax, r in zip(axes, show_df.itertuples()):
        ax.imshow(RESIZED[path_by_name[r.filename]])
        ax.set_title(f"true {r.true} | pred {r.predicted}\nconf {r.confidence:.2f} | {r.source}", fontsize=8)
        ax.axis("off")
    for ax in axes[len(show_df):]:
        ax.axis("off")
    fig.suptitle(f"{worst_model} ({exp_id(worst_model)}) - most confident misclassifications", y=1.02, fontsize=13)
    fig.tight_layout()
    save_fig(fig, "error_analysis", f"{exp_id(worst_model)}_misclassified_examples")
else:
    print(f"{worst_model} made no mistakes on this test set - nothing to show.")

## 15. Confidence & Reliability Analysis
Every prediction's full probability vector is saved, not just the final class (a neural network can be confident **and** wrong). Predictions are split into four buckets against the `CONFIDENCE_THRESHOLD = 0.90` gate:

```
Apple: 0.964
Banana: 0.036
Prediction: Apple
Confidence: 96.4%
```

In [ ]:
def reliability_counts(model_id):
    res = TEST_PREDS[model_id]
    conf = res["probs"][np.arange(len(res["preds"])), res["preds"]]
    correct = res["preds"] == res["labels"]
    high = conf >= CONFIDENCE_THRESHOLD
    return conf, correct, {
        "correct_high": int((correct & high).sum()), "correct_low": int((correct & ~high).sum()),
        "incorrect_high": int((~correct & high).sum()), "incorrect_low": int((~correct & ~high).sum()),
    }

reliability_rows = []
for m in MODEL_IDS:
    conf, correct, counts = reliability_counts(m)
    prob_banana = TEST_PREDS[m]["probs"][:, CLASSES.index("Banana")]
    auc = roc_auc_score(TEST_PREDS[m]["labels"], prob_banana) if len(set(TEST_PREDS[m]["labels"])) > 1 else float("nan")
    reliability_rows.append({"Model": m, **counts,
                             "mean_conf_correct": float(conf[correct].mean()) if correct.any() else float("nan"),
                             "mean_conf_incorrect": float(conf[~correct].mean()) if (~correct).any() else float("nan"),
                             "frac_conf_ge_thr": float((conf >= CONFIDENCE_THRESHOLD).mean()), "roc_auc": auc})
reliability_df = pd.DataFrame(reliability_rows)
print(reliability_df.round(3).to_string(index=False))
if (reliability_df["incorrect_high"] > 0).any():
    print("\nWARNING!!! 'incorrect_high' = confidently WRONG predictions - the most concerning bucket for a deployed confidence gate.")
reliability_df.to_csv(f"{DIRS['reliability']}/reliability_summary_{COMPUTER_ID}.csv", index=False)

# Illustrative single example, in the format shown above
_m, _i = MODEL_IDS[-1], 0
_probs = TEST_PREDS[_m]["probs"][_i]
print(f"\nExample ({_m}, {os.path.basename(test_paths[_i])}):")
for c, name in enumerate(CLASSES):
    print(f"  {name}: {_probs[c]:.3f}")
print(f"  Prediction: {CLASSES[_probs.argmax()]}")
print(f"  Confidence: {_probs.max():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
cat_names = ["correct_high", "correct_low", "incorrect_high", "incorrect_low"]
x = np.arange(len(MODEL_IDS)); width = 0.2
for i, cat in enumerate(cat_names):
    ax.bar(x + i * width, reliability_df[cat], width, label=cat)
ax.set_xticks(x + 1.5 * width); ax.set_xticklabels(MODEL_IDS)
ax.set_ylabel("test images"); ax.legend(fontsize=8, ncol=2)
fig.suptitle(f"Confidence x correctness, threshold={CONFIDENCE_THRESHOLD} ({COMPUTER_ID})", fontsize=13)
fig.tight_layout()
save_fig(fig, "reliability", f"reliability_breakdown_{COMPUTER_ID}")

## 16. Feature-Space Analysis [STRETCH]
```
Image → Model → Penultimate layer → Feature vector
```
Features are extracted from the layer just before the classification head of every model, then used to look at the Apple/Banana feature distributions, distance from class centroids, and whether low-confidence examples sit further from the known class distributions. These distances become one of the three signals for Section 18's unsupported-input strategy.

`RUN_STRETCH = False` skips this section (and 17–19) entirely.

In [ ]:
if RUN_STRETCH:
    def penultimate_features(model, model_id, x):
        """Return the penultimate-layer feature vector for a batch of raw 0-1 images."""
        if MODEL_CONFIGS[model_id]["type"] == "custom_cnn":
            return model.pool(model.features(x)).flatten(1)
        x = (x - model.mean) / model.std
        return model.pool(model.features(x)).flatten(1)

    @torch.no_grad()
    def extract_features(model_id, loader):
        model = TRAINED_MODELS[model_id]; model.eval()
        feats, labels = [], []
        for x, y in loader:
            feats.append(penultimate_features(model, model_id, x.to(device)).cpu().numpy())
            labels.append(y.numpy())
        return np.concatenate(feats), np.concatenate(labels)

    FEATURES = {}
    for m in MODEL_IDS:
        tr_f, tr_y = extract_features(m, train_eval_loader)
        te_f, te_y = extract_features(m, test_loader)
        FEATURES[m] = {"train_f": tr_f, "train_y": tr_y, "test_f": te_f, "test_y": te_y}
        np.savez_compressed(f"{DIRS['embeddings']}/{exp_id(m)}_embeddings.npz", train_f=tr_f, train_y=tr_y, test_f=te_f, test_y=te_y)
        print(f"{m}: {tr_f.shape[1]}-d penultimate features  |  train {tr_f.shape}  |  test {te_f.shape}")
else:
    print("RUN_STRETCH is off - Section 16 skipped.")

In [ ]:
if RUN_STRETCH:
    # Class centroids + a shared, shrinkage-regularised covariance (Ledoit-Wolf) from TRAIN features
    CLASS_STATS = {}
    for m in MODEL_IDS:
        tr_f, tr_y = FEATURES[m]["train_f"], FEATURES[m]["train_y"]
        centroids = {c: tr_f[tr_y == c].mean(axis=0) for c in range(len(CLASSES))}
        residuals = tr_f - np.stack([centroids[y] for y in tr_y])
        cov_inv = np.linalg.pinv(LedoitWolf().fit(residuals).covariance_)
        CLASS_STATS[m] = {"centroids": centroids, "cov_inv": cov_inv}

    def mahalanobis_distance(model_id, feats):
        """Distance from each feature vector to its NEAREST class centroid (shared covariance)."""
        stats = CLASS_STATS[model_id]
        dists = [np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", feats - c, stats["cov_inv"], feats - c), 0))
                 for c in stats["centroids"].values()]
        return np.stack(dists, axis=1).min(axis=1)

    # 2D PCA of the test features, for a visual check of class separation
    fig, axes = plt.subplots(1, len(MODEL_IDS), figsize=(4.3 * len(MODEL_IDS), 4))
    for ax, m in zip(axes, MODEL_IDS):
        pca = PCA(n_components=2, random_state=SEED).fit(FEATURES[m]["train_f"])
        proj = pca.transform(FEATURES[m]["test_f"])
        for c, name in enumerate(CLASSES):
            mask = FEATURES[m]["test_y"] == c
            ax.scatter(proj[mask, 0], proj[mask, 1], s=10, alpha=0.6, label=name)
        ax.set_title(f"{m}", fontsize=10); ax.legend(fontsize=7)
    fig.suptitle(f"Test-set feature space, penultimate layer, PCA to 2D ({COMPUTER_ID})", y=1.03, fontsize=13)
    fig.tight_layout()
    save_fig(fig, "embeddings", f"feature_space_pca_{COMPUTER_ID}")

    # Does low confidence line up with being far from the training class centroids?
    dist_rows = []
    for m in MODEL_IDS:
        d = mahalanobis_distance(m, FEATURES[m]["test_f"])
        res = TEST_PREDS[m]
        correct = res["preds"] == res["labels"]
        conf = res["probs"][np.arange(len(res["preds"])), res["preds"]]
        low_conf = conf < CONFIDENCE_THRESHOLD
        dist_rows.append({"Model": m, "mean_dist_correct": d[correct].mean(),
                          "mean_dist_incorrect": d[~correct].mean() if (~correct).any() else np.nan,
                          "mean_dist_low_conf": d[low_conf].mean() if low_conf.any() else np.nan,
                          "mean_dist_high_conf": d[~low_conf].mean()})
    dist_df = pd.DataFrame(dist_rows)
    print(dist_df.round(3).to_string(index=False))
    dist_df.to_csv(f"{DIRS['embeddings']}/feature_distance_summary_{COMPUTER_ID}.csv", index=False)
else:
    print("RUN_STRETCH is off - Section 16 (continued) skipped.")

#### Reasoning: Ledoit-Wolf Shrinkage
With ~1,780 training images spread across a 256-d (custom CNN) or 1,280-d (EfficientNetB0) feature space, a plain empirical covariance matrix is poorly conditioned (too few samples per dimension) and Mahalanobis distances computed from it are unstable. Ledoit-Wolf shrinkage regularises the covariance estimate toward a diagonal matrix, which is standard practice for exactly this small-sample regime and keeps the distance metric usable for Section 18's rejection signal.

## 17. Test-Time Augmentation [STRETCH]
```
Original → Prediction
Transformation 1 → Prediction
Transformation 2 → Prediction
Transformation 3 → Prediction
```
Checks whether each model predicts the same class for a test image and a few transformed versions of it. Low consistency is a reliability signal independent of the raw softmax confidence.

In [ ]:
if RUN_STRETCH:
    TTA_TRANSFORMS = {
        "original":   lambda img: eval_transform(img),
        "hflip":      lambda img: eval_transform(TF.hflip(img)),
        "rotate+10":  lambda img: eval_transform(TF.rotate(img, 10, interpolation=transforms.InterpolationMode.BILINEAR)),
        "rotate-10":  lambda img: eval_transform(TF.rotate(img, -10, interpolation=transforms.InterpolationMode.BILINEAR)),
        "brighter":   lambda img: eval_transform(TF.adjust_brightness(img, 1.25)),
    }
    TTA_SAMPLE_SIZE = 60 if not QUICK_TEST else 10

    @torch.no_grad()
    def tta_predict(model_id, n_samples=TTA_SAMPLE_SIZE):
        model = TRAINED_MODELS[model_id]; model.eval()
        rng = np.random.RandomState(SEED)
        idx = rng.choice(len(test_samples), size=min(n_samples, len(test_samples)), replace=False)
        rows = []
        for i in idx:
            path, true_label = test_samples[i]
            preds = {}
            for name, tf in TTA_TRANSFORMS.items():
                x = tf(RESIZED[path]).unsqueeze(0).to(device)
                probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
                preds[name] = (CLASSES[int(probs.argmax())], float(probs.max()))
            row = {"filename": os.path.basename(path), "true": CLASSES[true_label]}
            for name, (pred, conf) in preds.items():
                row[f"{name}_pred"], row[f"{name}_conf"] = pred, conf
            row["consistency"] = "HIGH" if len({p for p, _ in preds.values()}) == 1 else "LOW"
            rows.append(row)
        return pd.DataFrame(rows)

    TTA_RESULTS = {}
    for m in MODEL_IDS:
        df = tta_predict(m)
        TTA_RESULTS[m] = df
        low_frac = (df["consistency"] == "LOW").mean()
        print(f"{m}: {low_frac:.1%} of {len(df)} sampled test images have LOW consistency across {len(TTA_TRANSFORMS)} TTA views")
        df.to_csv(f"{DIRS['tta']}/{exp_id(m)}_tta.csv", index=False)

    _m, _row = MODEL_IDS[-1], TTA_RESULTS[MODEL_IDS[-1]].iloc[0]
    print(f"\nExample ({_m}, {_row['filename']}):")
    for name in TTA_TRANSFORMS:
        print(f"  {name:<10}: {_row[f'{name}_pred']} {_row[f'{name}_conf']:.0%}")
    print(f"  Consistency: {_row['consistency']}")
else:
    print("RUN_STRETCH is off - Section 17 skipped.")

## 18. Unsupported-Input / OOD Analysis [STRETCH]
```
Uploaded image
      ↓
Confidence + Feature-space similarity + Prediction consistency
      ↓
Reliable?  →  Accept / Uncertain
```
If an image cannot be confidently classified, the application should say: *"Unable to confidently classify this image. Please upload a clear image of an apple or banana."*

### 18.1 Out-of-Distribution Analysis

In [ ]:
if RUN_STRETCH:
    def combined_reliability(model_id):
        """Accept only when confidence clears the gate AND the feature-space distance is inside the
        95th percentile of the model's OWN training distances (never the test distances)."""
        res = TEST_PREDS[model_id]
        conf = res["probs"][np.arange(len(res["preds"])), res["preds"]]
        train_dist = mahalanobis_distance(model_id, FEATURES[model_id]["train_f"])
        test_dist = mahalanobis_distance(model_id, FEATURES[model_id]["test_f"])
        dist_threshold = float(np.percentile(train_dist, 95))
        tta_ok = TTA_RESULTS[model_id].set_index("filename")["consistency"] if model_id in TTA_RESULTS else None

        rows = []
        for i, path in enumerate(test_paths):
            fn = os.path.basename(path)
            consistency = tta_ok.get(fn, "not sampled") if tta_ok is not None else "not run"
            accept = bool(conf[i] >= CONFIDENCE_THRESHOLD and test_dist[i] <= dist_threshold and consistency != "LOW")
            rows.append({"filename": fn, "true": CLASSES[res["labels"][i]], "predicted": CLASSES[res["preds"][i]],
                        "confidence": float(conf[i]), "feature_distance": float(test_dist[i]),
                        "distance_threshold": dist_threshold, "tta_consistency": consistency, "accept": accept,
                        "message": "" if accept else "Unable to confidently classify this image. Please upload a clear image of an apple or banana."})
        return pd.DataFrame(rows)

    OOD_SIGNALS = {}
    for m in MODEL_IDS:
        df = combined_reliability(m)
        OOD_SIGNALS[m] = df
        print(f"{m}: {df['accept'].mean():.1%} of the Apple/Banana test set accepted by the combined signal")
        df.to_csv(f"{DIRS['ood']}/{exp_id(m)}_reliability_signal.csv", index=False)

    @torch.no_grad()
    def compute_signals(model_id, image, dist_threshold):
        """Confidence, feature-space distance and TTA consistency for ONE already-loaded PIL image -
        the exact same three checks combined_reliability() applies to the test set, so a genuine OOD
        image is judged by the identical rule a deployed app would use, not confidence alone."""
        model = TRAINED_MODELS[model_id]
        x = eval_transform(image).unsqueeze(0).to(device)
        probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
        conf = float(probs.max())
        dist = float(mahalanobis_distance(model_id, penultimate_features(model, model_id, x).cpu().numpy())[0])
        preds = {CLASSES[int(probs.argmax())]}
        for name, tf in TTA_TRANSFORMS.items():
            if name == "original":
                continue
            pt = torch.softmax(model(tf(image).unsqueeze(0).to(device)), dim=1)[0].cpu().numpy()
            preds.add(CLASSES[int(pt.argmax())])
        consistency = "HIGH" if len(preds) == 1 else "LOW"
        accept = bool(conf >= CONFIDENCE_THRESHOLD and dist <= dist_threshold and consistency != "LOW")
        return conf, dist, consistency, accept

    # Optional: validate the rejection mechanism against genuinely non-Apple/Banana images, if provided.
    OOD_EXAMPLES_DIR = "./OOD_Examples"
    OOD_META_PATH = os.path.join(OOD_EXAMPLES_DIR, "Metadata", "ood_metadata.csv")

    def resolve_ood_path(root, source_type, label, filename):
        """Find one OOD image file: try the expected <source_type>/<label>/<filename> layout
        first, then fall back to a recursive search by filename alone, so a folder that doesn't
        exactly match the CSV's source_type/label (different casing, renamed category, ...)
        still resolves instead of silently - or crashingly - being treated as missing."""
        expected = os.path.join(root, str(source_type), str(label), str(filename))
        if os.path.exists(expected):
            return expected, True
        for dirpath, _, files in os.walk(root):
            if filename in files:
                return os.path.join(dirpath, filename), False
        return None, False

    HAVE_OOD = False
    if os.path.exists(OOD_META_PATH):
        ood_meta = pd.read_csv(OOD_META_PATH)
        resolved = ood_meta.apply(
            lambda r: resolve_ood_path(OOD_EXAMPLES_DIR, r["source_type"], r["label"], r["filename"]), axis=1)
        ood_meta["path"] = [p for p, _ in resolved]
        ood_meta["path_matched_metadata"] = [mm for _, mm in resolved]
        found = ood_meta["path"].notna()
        n_fallback = int((found & ~ood_meta["path_matched_metadata"]).sum())

        if (~found).any():
            print(f"WARNING!!! {(~found).sum()} rows in '{OOD_META_PATH}' have no matching file anywhere under "
                  f"'{OOD_EXAMPLES_DIR}' - skipped:")
            for fn in ood_meta.loc[~found, "filename"].head(5):
                print("   ", fn)
        if n_fallback:
            print(f"WARNING!!! {n_fallback} images were found, but not at the <source_type>/<label> path their metadata "
                  f"row implies - the CSV's source_type/label for those rows may not match the actual folder.")
        ood_meta = ood_meta[found].reset_index(drop=True)

        if len(ood_meta) == 0:
            print(f"\nNo OOD images could be located under '{OOD_EXAMPLES_DIR}' - check that the filenames in "
                  f"'{OOD_META_PATH}' match files somewhere in that folder. Skipping OOD validation for this run.")
        else:
            print(f"\nFound {len(ood_meta)} genuine OOD images in '{OOD_EXAMPLES_DIR}' "
                  f"({ood_meta['source_type'].nunique()} sources x {ood_meta['label'].nunique()} categories) - "
                  f"checking all three signals (confidence + feature distance + TTA consistency), same as the test set")

            OOD_DIST_THRESHOLD = {}
            for m in MODEL_IDS:
                dist_thr = float(np.percentile(mahalanobis_distance(m, FEATURES[m]["train_f"]), 95))
                OOD_DIST_THRESHOLD[m] = dist_thr
                conf_l, dist_l, cons_l, acc_l = [], [], [], []
                for r in ood_meta.itertuples():
                    img = Image.open(r.path).convert("RGB").resize(IMAGE_SIZE, Image.BICUBIC)
                    conf, dist, consistency, accept = compute_signals(m, img, dist_thr)
                    conf_l.append(conf); dist_l.append(dist); cons_l.append(consistency); acc_l.append(accept)
                ood_meta[f"{m}_confidence"] = conf_l
                ood_meta[f"{m}_feature_distance"] = dist_l
                ood_meta[f"{m}_tta_consistency"] = cons_l
                ood_meta[f"{m}_wrongly_accepted"] = acc_l

                n_wrong = sum(acc_l)
                print(f"\n{m}: {n_wrong}/{len(ood_meta)} genuine OOD images wrongly accepted as confident predictions "
                      f"({n_wrong / len(ood_meta):.1%})")
                breakdown = ood_meta.groupby(["source_type", "label"])[f"{m}_wrongly_accepted"].agg(["sum", "count"])
                breakdown.columns = ["wrongly_accepted", "n"]
                breakdown["rate"] = (breakdown["wrongly_accepted"] / breakdown["n"]).round(3)
                print(breakdown.to_string())
            ood_meta.to_csv(f"{DIRS['ood']}/ood_examples_results_{COMPUTER_ID}.csv", index=False)
            HAVE_OOD = True
    elif os.path.isdir(OOD_EXAMPLES_DIR):
        print(f"\n'{OOD_EXAMPLES_DIR}' exists but no metadata CSV at '{OOD_META_PATH}' - expected columns: "
              f"image_id, filename, label, source_type, source_name, licence, creator, platform, format")
    else:
        print(f"\nNo '{OOD_EXAMPLES_DIR}' folder found - the acceptance rates above describe Apple/Banana test images only.")
else:
    print("RUN_STRETCH is off - Section 18 skipped.")
    HAVE_OOD = False

### 18.2 Robustness under extreme corruption
Two specific stress tests, not the mild, realistic transforms Section 17 uses for prediction-consistency: rotating a test image by exactly 90° (a clean multiple of 90° leaves no black-corner fill on a square image, unlike an arbitrary angle), and replacing 80% of its pixels with random noise. These represent two different kinds of "bad photo" a real camera can produce, and a well-behaved system should treat them differently:

- **90° rotation keeps every pixel of information**, just reoriented - a fridge photo taken sideways. Accuracy **should** hold up close to the unperturbed test accuracy; a large drop means the model has learned canonical orientation rather than the fruit itself.
- **Noise destroys information, more of it as the level rises.** There is no single "correct" percentage to test - a single point either sits in the easy, uninteresting regime (even a poorly-calibrated model rejects near-pure noise) or the equally uninteresting one (a trivial amount of noise should never move confidence much). What is actually informative is **where along that range confidence crosses the gate**, so this sweeps {10, 30, 50, 70, 90}% rather than testing one fixed value, and plots the resulting curve. A well-calibrated model's confidence should fall steadily as the noise level rises, crossing below `CONFIDENCE_THRESHOLD` well before 90%; a model that stays confident even at 90% noise has a miscalibrated gate.

In [ ]:
if RUN_STRETCH:
    def rotate_exact(img, degrees):
        """Rotate by an exact multiple of 90 - on a square image this leaves no black-corner fill,
        unlike an arbitrary angle (compare Section 5's training-time RandomRotation)."""
        return img.rotate(degrees, expand=False)

    def heavy_noise(img, level, rng=None):
        """Replace `level` fraction of pixels with uniform random colour - a severe corruption stress test."""
        rng = rng or np.random.RandomState(SEED)
        arr = np.asarray(img).copy()
        mask = rng.rand(*arr.shape[:2]) < level
        arr[mask] = rng.randint(0, 256, size=(mask.sum(), 3))
        return Image.fromarray(arr)

    NOISE_LEVELS = [0.1, 0.3, 0.5, 0.7, 0.9]    # a sweep, not one arbitrary point - see the reasoning above

    @torch.no_grad()
    def rotation_predict(model_id, n_samples=TTA_SAMPLE_SIZE):
        model = TRAINED_MODELS[model_id]; model.eval()
        rng = np.random.RandomState(SEED)
        idx = rng.choice(len(test_samples), size=min(n_samples, len(test_samples)), replace=False)
        rows = []
        for i in idx:
            path, true_label = test_samples[i]
            row = {"filename": os.path.basename(path), "true": CLASSES[true_label]}
            for name, img in [("original", RESIZED[path]), ("rotate_90", rotate_exact(RESIZED[path], 90))]:
                x = eval_transform(img).unsqueeze(0).to(device)
                probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
                row[f"{name}_pred"], row[f"{name}_conf"] = CLASSES[int(probs.argmax())], float(probs.max())
            rows.append(row)
        return pd.DataFrame(rows)

    @torch.no_grad()
    def noise_sweep_predict(model_id, levels=NOISE_LEVELS, n_samples=TTA_SAMPLE_SIZE):
        model = TRAINED_MODELS[model_id]; model.eval()
        rng = np.random.RandomState(SEED)
        idx = rng.choice(len(test_samples), size=min(n_samples, len(test_samples)), replace=False)
        rows = []
        for i in idx:
            path, true_label = test_samples[i]
            for level in levels:
                img = heavy_noise(RESIZED[path], level, rng=np.random.RandomState(SEED))
                x = eval_transform(img).unsqueeze(0).to(device)
                probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
                rows.append({"filename": os.path.basename(path), "true": CLASSES[true_label], "noise_level": level,
                            "pred": CLASSES[int(probs.argmax())], "confidence": float(probs.max())})
        return pd.DataFrame(rows)

    ROTATION_RESULTS, NOISE_SWEEP_RESULTS = {}, {}
    for m in MODEL_IDS:
        rdf = rotation_predict(m)
        ROTATION_RESULTS[m] = rdf
        orig_acc = (rdf["original_pred"] == rdf["true"]).mean()
        rot_acc = (rdf["rotate_90_pred"] == rdf["true"]).mean()
        rot_conf_drop = rdf["original_conf"].mean() - rdf["rotate_90_conf"].mean()
        print(f"{m}: 90-degree rotation -> accuracy {rot_acc:.1%} (vs {orig_acc:.1%} unperturbed), "
              f"mean confidence drop {rot_conf_drop:+.3f}")
        rdf.to_csv(f"{DIRS['ood']}/{exp_id(m)}_rotation_robustness.csv", index=False)

        ndf = noise_sweep_predict(m)
        NOISE_SWEEP_RESULTS[m] = ndf
        summary = ndf.groupby("noise_level").apply(lambda g: pd.Series({
            "accuracy": (g["pred"] == g["true"]).mean(), "mean_confidence": g["confidence"].mean(),
            "frac_rejected": (g["confidence"] < CONFIDENCE_THRESHOLD).mean()}))
        print(f"    noise sweep:\n{summary.round(3).to_string().replace(chr(10), chr(10) + '    ')}")
        ndf.to_csv(f"{DIRS['ood']}/{exp_id(m)}_noise_sweep.csv", index=False)

    # Degradation curves: mean confidence and rejection rate vs noise level, one line per model
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for m in MODEL_IDS:
        by_level = NOISE_SWEEP_RESULTS[m].groupby("noise_level")
        conf_mean = by_level["confidence"].mean()
        rej_rate = by_level.apply(lambda g: (g["confidence"] < CONFIDENCE_THRESHOLD).mean())
        axes[0].plot(conf_mean.index, conf_mean.values, marker="o", label=m)
        axes[1].plot(rej_rate.index, rej_rate.values, marker="o", label=m)
    axes[0].axhline(CONFIDENCE_THRESHOLD, color="grey", linestyle="--", linewidth=1, label="gate")
    axes[0].set_xlabel("noise level"); axes[0].set_ylabel("mean confidence"); axes[0].legend(fontsize=8)
    axes[1].set_xlabel("noise level"); axes[1].set_ylabel("fraction rejected"); axes[1].legend(fontsize=8)
    fig.suptitle(f"Confidence degradation under noise ({COMPUTER_ID})", y=1.03, fontsize=13)
    fig.tight_layout()
    save_fig(fig, "ood", f"noise_sweep_{COMPUTER_ID}")
else:
    print("RUN_STRETCH is off - robustness stress test skipped.")

### 18.3 Finding the cutoff
The point of combining three signals is to decide, automatically, when to show *"Unable to confidently classify this image..."* instead of a prediction. `CONFIDENCE_THRESHOLD` (Section 1) and the 95th-percentile feature-distance cutoff (Section 18 above) are the values currently wired into that decision - this section checks whether they are actually good cutoffs, using the one thing that can answer that: real in-domain images (the test set, which should be **accepted**) next to real OOD images (`OOD_Examples`, which should be **rejected**).

For confidence and feature distance (both continuous), an ROC curve of "is this genuinely OOD" against the signal gives the best achievable cutoff at this model's current features (Youden's J: the point on the curve furthest above the diagonal) - marked on each plot next to the currently configured value, so the two can be compared directly. TTA consistency is binary (HIGH/LOW), so it gets a plain rejection-rate comparison instead of a curve.

In [ ]:
if RUN_STRETCH and HAVE_OOD:
    cutoff_rows = []
    fig, axes = plt.subplots(len(MODEL_IDS), 2, figsize=(9.5, 3.3 * len(MODEL_IDS)))
    if len(MODEL_IDS) == 1:
        axes = axes.reshape(1, -1)

    for row_i, m in enumerate(MODEL_IDS):
        test_conf = TEST_PREDS[m]["probs"].max(axis=1)
        test_dist = mahalanobis_distance(m, FEATURES[m]["test_f"])
        ood_conf = ood_meta[f"{m}_confidence"].values
        ood_dist = ood_meta[f"{m}_feature_distance"].values

        # y = 1 means "should be REJECTED" (genuine OOD); y = 0 means "should be ACCEPTED" (test set)
        y = np.concatenate([np.zeros(len(test_conf)), np.ones(len(ood_conf))])

        for col, (label, score, configured, invert) in enumerate([
            ("confidence", np.concatenate([1 - test_conf, 1 - ood_conf]), CONFIDENCE_THRESHOLD, True),
            ("feature distance", np.concatenate([test_dist, ood_dist]), OOD_DIST_THRESHOLD[m], False),
        ]):
            fpr, tpr, thr = roc_curve(y, score)
            j = tpr - fpr
            best = int(j.argmax())
            auc = float(roc_auc_score(y, score))
            recommended = (1 - thr[best]) if invert else thr[best]

            ax = axes[row_i, col]
            ax.plot(fpr, tpr, label=f"AUC={auc:.2f}")
            ax.plot([0, 1], [0, 1], linestyle=":", color="grey", linewidth=1)
            ax.scatter([fpr[best]], [tpr[best]], color="#c0392b", zorder=5, label=f"best cutoff = {recommended:.3f}")
            ax.set_title(f"{m} - {label}", fontsize=10)
            ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate"); ax.legend(fontsize=7)

            cutoff_rows.append({"model": m, "signal": label, "configured_cutoff": configured,
                                "recommended_cutoff": round(recommended, 4), "auc": round(auc, 4)})

        # TTA consistency: plain rejection-rate comparison (binary signal, no ROC)
        test_low = (TTA_RESULTS[m]["consistency"] == "LOW").mean() if m in TTA_RESULTS else float("nan")
        ood_low = (ood_meta[f"{m}_tta_consistency"] == "LOW").mean()
        cutoff_rows.append({"model": m, "signal": "tta_consistency", "configured_cutoff": "!= LOW",
                            "recommended_cutoff": f"test {test_low:.1%} LOW vs OOD {ood_low:.1%} LOW", "auc": None})

    fig.suptitle(f"Finding the cutoff: in-domain test set vs genuine OOD ({COMPUTER_ID})", y=1.0, fontsize=13)
    fig.tight_layout()
    save_fig(fig, "ood", f"cutoff_analysis_{COMPUTER_ID}")

    cutoff_df = pd.DataFrame(cutoff_rows)
    print(cutoff_df.to_string(index=False))
    print(f"\nCurrently configured: CONFIDENCE_THRESHOLD = {CONFIDENCE_THRESHOLD}; feature-distance cutoff = "
          f"95th percentile of each model's own training distances (values above, per model).")
    print("A 'recommended' cutoff far from the configured one means the fixed value is costing real separation on")
    print("this OOD sample - worth revisiting Section 1's CONFIDENCE_THRESHOLD or the distance percentile in Section 18.")
    cutoff_df.to_csv(f"{DIRS['ood']}/cutoff_analysis_{COMPUTER_ID}.csv", index=False)
elif RUN_STRETCH:
    print("No OOD_Examples found - the cutoff can only be evaluated against genuine OOD images (see the Critical")
    print("Limitation below), so this analysis is skipped until OOD_Examples/Metadata/ood_metadata.csv exists.")
else:
    print("RUN_STRETCH is off - cutoff analysis skipped.")

## 19. Computational Comparison [STRETCH]
Parameter count, checkpoint size, training time, CPU inference time and (where available) peak memory – another dimension beyond "which model has the highest accuracy".

In [ ]:
if RUN_STRETCH:
    @torch.no_grad()
    def measure_inference_time(model_id, num_runs=50):
        model = TRAINED_MODELS[model_id]; model.eval()
        x = next(iter(test_loader))[0][:1].to(device)      # single image, like a live prediction
        for _ in range(5):
            model(x)                                        # warm-up
        t0 = time.time()
        for _ in range(num_runs):
            model(x)
        return (time.time() - t0) / num_runs * 1000          # ms per image

    comp_rows = []
    for m in MODEL_IDS:
        meta = RESULTS[m]
        comp_rows.append({
            "Model": m, "Params (total)": meta["n_params_total"], "Params (trainable)": meta["n_params_trainable"],
            "Checkpoint (MB)": os.path.getsize(meta["checkpoint_path"]) / (1024 * 1024),
            "Train time (s)": meta["train_time_s"], "Epochs run": meta["epochs"],
            "Inference (ms/image, CPU)": measure_inference_time(m),
        })
    comp_df = pd.DataFrame(comp_rows)
    print(comp_df.round(2).to_string(index=False))
    comp_df.to_csv(f"{DIRS['computational']}/computational_comparison_{COMPUTER_ID}.csv", index=False)

    if psutil is not None:
        rss_mb = psutil.Process().memory_info().rss / 1e6
        print(f"\nCurrent process memory (RSS): {rss_mb:.0f} MB - rough indicator only, all four models are")
        print("loaded simultaneously at this point in the notebook, which a deployed single-model app would not do.")
else:
    print("RUN_STRETCH is off - Section 19 skipped.")

## 20. Save & Export All Results
Every result produced above already lives under `results/<subfolder>/` (or `results_QUICKTEST/` in a smoke test). This section adds one model-metadata card per model and one manifest for the whole run, then zips the folder for easy sending.

```
results/
├── metrics/                 (per-model + combined test metrics, student-vs-public)
├── confusion_matrices/
├── learning_curves/
├── predictions/             (reserved - raw predictions live inside metrics/ and error_analysis/)
├── embeddings/               [stretch]
├── experiment_logs/          (one JSON per experiment_id: hyperparameters, history, environment)
├── reproducibility/          (Model x Computer tables)
├── checkpoints/              (.pt weights, one per experiment_id)
├── error_analysis/
├── reliability/
├── tta/                      [stretch]
├── ood/                      [stretch]
├── computational/            [stretch]
└── audit/                    (Section 3 evidence)
```

In [ ]:
metrics_by_model = metrics_df.set_index("Model")
for m in MODEL_IDS:
    meta = RESULTS[m]
    card = {
        "ModelVersion": f"{m}_{COMPUTER_ID}_{NOTEBOOK_VERSION}", "ModelType": MODEL_CONFIGS[m]["type"],
        "ExperimentID": exp_id(m), "ComputerID": COMPUTER_ID, "DatasetVersion": DATASET_VERSION,
        "PreprocessingVersion": PREPROCESSING_VERSION, "TrainingDate": meta["environment"]["timestamp"],
        "ImageSize": list(IMAGE_SIZE), "BatchSize": BATCH_SIZE, "Hyperparameters": MODEL_CONFIGS[m],
        "FrameworkVersion": f"PyTorch {torch.__version__}",
        "ParamsTotal": meta["n_params_total"], "ParamsTrainable": meta["n_params_trainable"],
        "TestAccuracy": float(metrics_by_model.loc[m, "Accuracy"]), "TestF1": float(metrics_by_model.loc[m, "F1"]),
    }
    with open(f"{DIRS['experiment_logs']}/{exp_id(m)}_model_card.json", "w") as f:
        json.dump(card, f, indent=2, default=str)

manifest = {
    "computer_id": COMPUTER_ID, "notebook_version": NOTEBOOK_VERSION, "dataset_version": DATASET_VERSION,
    "dataset_fingerprint": DATASET_FINGERPRINT, "config_hash": CONFIG_HASH, "audit_status": AUDIT_STATUS,
    "m2_lock_loaded": m2_lock is not None, "m2_config": MODEL_CONFIGS["M2"],
    "quick_test": QUICK_TEST, "run_stretch": RUN_STRETCH, "generated": datetime.datetime.now().isoformat(timespec="seconds"),
    "experiment_ids": [exp_id(m) for m in MODEL_IDS],
    "files": sorted(os.path.relpath(p, RESULTS_ROOT) for p in glob.glob(f"{RESULTS_ROOT}/**/*", recursive=True) if os.path.isfile(p)),
}
with open(f"{RESULTS_ROOT}/manifest_{COMPUTER_ID}.json", "w") as f:
    json.dump(manifest, f, indent=2)

zip_path = shutil.make_archive(f"results_{COMPUTER_ID}", "zip", RESULTS_ROOT)
print(f"Results saved: YES  ({len(manifest['files'])} files under {RESULTS_ROOT}/)")
print(f"Models saved : YES  ({len(MODEL_IDS)} checkpoints in {DIRS['checkpoints']}/)")
print(f"Zipped to    : {zip_path}")

## 21. Final Experiment Summary

In [ ]:
print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"Computer: {COMPUTER_ID}")
print(f"Models  : {len(MODEL_IDS)}")
for m in MODEL_IDS:
    print(f"  {m}: {MODEL_CONFIGS[m]['name']}")
print(f"\nDataset          : {DATASET_VERSION} ({EXPECTED['total']:,} images)  |  fingerprint {DATASET_FINGERPRINT}")
print(f"Image size       : {IMAGE_SIZE[0]} x {IMAGE_SIZE[1]}  |  Batch size: {BATCH_SIZE}")
print(f"Test set         : {len(test_labels)} images")
print(f"Audit status     : {AUDIT_STATUS}")
print(f"Config hash      : {CONFIG_HASH}")
print(f"Quick test       : {QUICK_TEST}  |  Stretch sections run: {RUN_STRETCH}")
print()
print(metrics_df[["Model", "Accuracy", "Precision", "Recall", "F1"]].round(4).to_string(index=False))
print()
print("Results saved        : YES")
print("Models saved         : YES")
print("Experiment logs saved: YES")
print()
print("This notebook does NOT declare a final winner. Model selection is an evidence-based decision made in")
print("Results_Collation_and_Comparison.ipynb, using the exported results from all four computers.")
print()
print(f"Next step: save this executed notebook as Model_Training_and_Experiments_{COMPUTER_ID}.ipynb")
print(f"and send it together with 'results_{COMPUTER_ID}.zip' (Section 20).")